# Phase 22: Hình v11 (hệ thống thiết kế mới, JNE)

Render các hình đã làm lại theo `paper/S2I_figure_plan_v11.md`: rộng **150 mm**, chữ **≥ 8 pt**, trục đặt theo mm, tiêu đề ngắn, không phụ đề. Hiện có **Fig. 1** và **Fig. 2**; các hình còn lại sẽ được thêm vào module `s2i/figs11.py` lần lượt.

- CPU là đủ, mỗi hình vài giây.
- Mỗi hình xuất `PDF` (chèn LaTeX), `SVG`, `PNG 600 dpi` và `<tên>_numbers.json` (mọi số in trên hình, kèm kết quả kiểm tra bố cục).
- Thư mục output: `MyDrive/iEEG_S2I/paper/figs11/` (không ghi đè hình cũ).

**Lưu ý về khoảng tin cậy:** bootstrap giờ dùng seed cố định theo tên từng phép so sánh, nên CI có thể lệch ±0,001 so với bản thảo. Sau khi đủ các hình, toàn bộ CI trong bản thảo sẽ được cập nhật từ cùng một nguồn.

In [ ]:
#@title 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
RUNS_DIR = f'{ROOT}/runs'
FIG_DIR  = f'{ROOT}/paper/figs11'
os.makedirs(f'{CODE_DIR}/s2i', exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.chdir(CODE_DIR)
!apt-get -qq install -y fonts-liberation > /dev/null
!pip -q install nilearn
print('OK')

## 2. Ghi code lên Drive

In [ ]:
%%writefile {CODE_DIR}/s2i/figstyle.py
"""
Shared figure style for the paper (Phase 10, v2). Every figure module imports from here.

Design rules (decided 12 Oct 2026, figure_plan.md section 5):
  * final print size: double column 180 mm, single column 88 mm; text 6-8 pt; one sans family
  * no gridlines: 1-2 reference lines with a meaning (zero, baseline mean) instead
  * range frame: spines run from the first to the last tick and sit 3 pt away from the data
  * panel headline = a short conclusion in ink, left-aligned with the axis; details go to the caption
  * colour: warm ink, muted ink-blue for our model, one earth-orange accent per figure, warm greys
  * patients: deterministic beeswarm or paired lines, never random jitter
  * labels are placed next to the data; positions that need hand tuning are read from NUDGE[fig][key]
"""
import glob
import os

import numpy as np
import matplotlib as mpl

mpl.use("Agg")
import matplotlib.pyplot as plt                                    # noqa: E402
import matplotlib.transforms as mtransforms                        # noqa: E402
from matplotlib import font_manager                                # noqa: E402

MM = 1 / 25.4
PT = 1 / 72
W2, W1 = 180 * MM, 88 * MM

# ----------------------------------------------------------------------------- palette
INK, INK2, INK3 = "#1d1d1b", "#55534e", "#8d8a83"   # text / secondary text, ticks / tertiary notes
PATIENT = "#cfcac1"                                  # individual patients (background)
BASE = "#7d7a73"                                     # baselines
FIXED = "#9dbad6"                                    # fixed untrained filter bank
OURS = "#1f4e79"                                     # our model
OURS_SOFT = "#dce6f0"                                # band / fill for our model
ACCENT = "#c4512b"                                   # the one thing to look at (one per figure)
ACCENT_SOFT = "#f6e1d8"
EXT = "#2f7d6d"                                      # external cohort (figure 7)
WASH = "#f3efe8"                                     # warm background band instead of a grid
RULE = "#bdb8ae"                                     # reference lines

LBL_FIXED = "fixed untrained bank"


STYLE = "own"                                        # "own" (rules above) or "sp" (SciencePlots base)


def setup(style="own"):
    global STYLE
    STYLE = style
    for d in ("/usr/share/fonts/truetype/liberation", "/usr/share/fonts/truetype/liberation2"):
        for f in glob.glob(os.path.join(d, "LiberationSans*.ttf")):
            font_manager.fontManager.addfont(f)
    if style == "sp":
        # SciencePlots: boxed axes, inward mirrored ticks, minor ticks, 7 pt sans (nature). No LaTeX needed.
        import scienceplots                                        # noqa: F401  (registers the styles)
        plt.style.use(["science", "no-latex", "nature"])
        mpl.rcParams.update({"savefig.bbox": "standard", "savefig.dpi": 600, "pdf.fonttype": 42,
                             "svg.fonttype": "none", "axes.unicode_minus": True, "axes.titlelocation": "left",
                             "figure.dpi": 150, "text.color": INK})
        return
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "Liberation Sans", "DejaVu Sans"],
        "font.size": 6.5, "axes.labelsize": 6.5, "axes.titlesize": 7, "xtick.labelsize": 6,
        "ytick.labelsize": 6, "legend.fontsize": 6,
        "text.color": INK, "axes.labelcolor": INK, "axes.edgecolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
        "axes.linewidth": 0.5, "xtick.major.width": 0.5, "ytick.major.width": 0.5,
        "xtick.major.size": 2.0, "ytick.major.size": 2.0, "xtick.minor.size": 1.2, "ytick.minor.size": 1.2,
        "xtick.major.pad": 1.8, "ytick.major.pad": 1.8, "axes.labelpad": 2.5, "xtick.direction": "out",
        "ytick.direction": "out", "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
        "axes.titlepad": 5, "axes.titlelocation": "left", "legend.frameon": False,
        "axes.unicode_minus": True, "pdf.fonttype": 42, "svg.fonttype": "none", "savefig.dpi": 600,
        "lines.solid_capstyle": "round", "lines.dash_capstyle": "butt", "figure.dpi": 150,
        "mathtext.fontset": "custom", "mathtext.rm": "sans", "mathtext.it": "sans:italic",
        "mathtext.bf": "sans:bold", "mathtext.sf": "sans",
    })


# ----------------------------------------------------------------------------- axes helpers
def range_frame(ax, x=True, y=True, offset=3):
    """Spines from the first to the last tick, pushed `offset` pt away from the data (no-op in SciencePlots mode)."""
    if STYLE == "sp":
        return
    ax.figure.canvas.draw_idle()
    if y:
        t = [v for v in ax.get_yticks() if ax.get_ylim()[0] - 1e-9 <= v <= ax.get_ylim()[1] + 1e-9]
        if t:
            ax.spines["left"].set_bounds(t[0], t[-1])
        ax.spines["left"].set_position(("outward", offset))
    if x:
        t = [v for v in ax.get_xticks() if ax.get_xlim()[0] - 1e-9 <= v <= ax.get_xlim()[1] + 1e-9]
        if t:
            ax.spines["bottom"].set_bounds(t[0], t[-1])
        ax.spines["bottom"].set_position(("outward", offset))


def bare_y(ax):
    """Category axis on the left: no spine, no ticks."""
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0, pad=3)


def headline(ax, s, tag=None):
    """Short conclusion above the axis, left-aligned with the axis (ink, not grey). `tag` = one line of context
    (level, normalisation...) in small grey type between the headline and the axis."""
    if tag:
        tr = mtransforms.offset_copy(ax.transAxes, fig=ax.figure, x=0, y=2.5, units="points")
        ax.text(0.0, 1.0, tag, transform=tr, fontsize=5.6, color=INK3, va="bottom", ha="left")
    ax.set_title(s, loc="left", color=INK, fontsize=7, pad=11.5 if tag else 5)


def letters(fig, items):
    """Panel letters aligned with the outer edge of each axis' tick labels, top-aligned with its headline.
    items: list of (ax, 'a', dx_mm). dx_mm nudges horizontally (+ right)."""
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    W, H = fig.get_size_inches()
    for ax, s, dx in items:
        bb = ax.get_tightbbox(r).transformed(fig.transFigure.inverted())
        t = ax._left_title if ax._left_title.get_text() else ax.title
        title = t.get_window_extent(r).transformed(fig.transFigure.inverted())
        fig.text(bb.x0 + dx * MM / W, title.y1 + 0.4 * MM / H, s, fontsize=8, fontweight="bold", va="top",
                 ha="left", color=INK)


def ref_line(ax, v, axis="y", **kw):
    st = dict(color=RULE, lw=0.5, ls=(0, (2, 2)), zorder=0)
    st.update(kw)
    (ax.axhline if axis == "y" else ax.axvline)(v, **st)


def beeswarm(v, size_data, max_off=0.42):
    """Deterministic 1-D beeswarm: offsets (in category units) so that points `size_data` apart do not overlap."""
    v = np.asarray(v, float)
    order = np.argsort(v)
    off = np.zeros(len(v))
    placed = []
    step = 0.11
    for i in order:
        for k in range(0, 40):
            o = (k + 1) // 2 * step * (1 if k % 2 else -1) if k else 0.0
            if abs(o) > max_off:
                o = np.sign(o) * max_off
                break
            if all(abs(v[i] - v[j]) >= size_data or abs(o - off[j]) >= step * 0.95 for j in placed):
                break
        off[i] = o
        placed.append(i)
    return off


def boot_ci(v, n=10000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    m = v[rng.integers(0, len(v), (n, len(v)))].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def fmt(v, d=3, sign=False):
    s = f"{v:+.{d}f}" if sign else f"{v:.{d}f}"
    return s.replace("-", "\u2212")


def bracket(ax, x, y0, y1, text, color=INK, w=0.06, pad=0.05, fs=6.5, weight="bold", side="right", sub=None):
    """Vertical bracket between y0 and y1 at x (data), with a label beside it."""
    s = 1 if side == "right" else -1
    ax.plot([x, x + s * w, x + s * w, x], [y0, y0, y1, y1], color=color, lw=0.6, solid_joinstyle="miter",
            clip_on=False)
    ym = (y0 + y1) / 2
    ax.text(x + s * (w + pad), ym, text, color=color, fontsize=fs, fontweight=weight, va="center",
            ha="left" if s > 0 else "right", clip_on=False)
    if sub:
        ax.text(x + s * (w + pad), ym, "\n\n" + sub, color=INK2, fontsize=5.8, va="center",
                ha="left" if s > 0 else "right", clip_on=False, linespacing=1.0)


def _sp_box(fig):
    """SciencePlots mode: every visible axis gets its full box back; category axes (major tick length 0) lose the
    automatic minor ticks that would fall between categories."""
    from matplotlib.ticker import NullLocator
    for ax in fig.axes:
        if not ax.axison:
            continue
        for sp in ax.spines.values():
            sp.set_visible(True)
        for axis in (ax.xaxis, ax.yaxis):
            t = axis.get_major_ticks()
            if not t or t[0].tick1line.get_markersize() == 0 or not len(axis.get_majorticklocs()):
                axis.set_minor_locator(NullLocator())


def save(fig, out, name):
    os.makedirs(out, exist_ok=True)
    if STYLE == "sp":
        _sp_box(fig)
    for ext in ("pdf", "svg", "png"):
        fig.savefig(os.path.join(out, f"{name}.{ext}"))
    plt.close(fig)
    print("  ->", os.path.join(out, name + ".{pdf,svg,png}"))

In [ ]:
%%writefile {CODE_DIR}/s2i/render3d.py
"""
Minimal 3D rendering for the paper schematic (Fig 1): a fsaverage5 cortex drawn with matplotlib only.

Orthographic camera, smooth (vertex-normal) Lambert + weak specular shading, painter's sort, one rasterized
PolyCollection; everything else in the figure stays vector. Brain surface: fsaverage5 pial + sulcal depth
(FreeSurfer, via nilearn), cached as .npz after the first call.
"""
import os

import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.colors import to_rgb


def mix(c0, c1, t):
    """Linear blend of two colours, t in [0, 1] (scalar or array) -> RGB array."""
    c0, c1 = np.array(to_rgb(c0)), np.array(to_rgb(c1))
    t = np.clip(np.asarray(t, float), 0, 1)[..., None]
    return c0 * (1 - t) + c1 * t


def _unit(a):
    a = np.asarray(a, float)
    return a / np.linalg.norm(a)


def camera(az, el):
    """Rows: screen-right, screen-up, toward-viewer, in world (RAS) coordinates. az from +x toward +y, el up."""
    az, el = np.radians(az), np.radians(el)
    f = np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)])
    r = np.cross([0.0, 0.0, 1.0], f)
    r /= np.linalg.norm(r)
    return np.stack([r, np.cross(f, r), f])


def face_normals(v, f):
    n = np.cross(v[f[:, 1]] - v[f[:, 0]], v[f[:, 2]] - v[f[:, 0]])
    return n / (np.linalg.norm(n, axis=1, keepdims=True) + 1e-12)


def vertex_normals(v, f):
    fn = np.cross(v[f[:, 1]] - v[f[:, 0]], v[f[:, 2]] - v[f[:, 0]])     # area-weighted
    vn = np.zeros_like(v)
    for k in range(3):
        np.add.at(vn, f[:, k], fn)
    return vn / (np.linalg.norm(vn, axis=1, keepdims=True) + 1e-12)


def subdivide(v, f, vn, vals=None):
    """One midpoint subdivision (4 triangles per face); normals and per-vertex values are interpolated."""
    e = np.sort(np.r_[f[:, [0, 1]], f[:, [1, 2]], f[:, [2, 0]]], axis=1)
    ue, inv = np.unique(e, axis=0, return_inverse=True)
    inv = inv.ravel()
    nv = len(v)
    v2 = np.r_[v, (v[ue[:, 0]] + v[ue[:, 1]]) / 2]
    n2 = np.r_[vn, vn[ue[:, 0]] + vn[ue[:, 1]]]
    n2 /= np.linalg.norm(n2, axis=1, keepdims=True) + 1e-12
    m = nv + inv.reshape(3, -1).T                                         # midpoints of edges 01, 12, 20
    a, b, c = f[:, 0], f[:, 1], f[:, 2]
    f2 = np.r_[np.c_[a, m[:, 0], m[:, 2]], np.c_[m[:, 0], b, m[:, 1]], np.c_[m[:, 2], m[:, 1], c],
               np.c_[m[:, 0], m[:, 1], m[:, 2]]]
    out = [v2, f2, n2]
    if vals is not None:
        out.append(np.r_[vals, (vals[ue[:, 0]] + vals[ue[:, 1]]) / 2])
    return out


LIGHTS = (  # camera-space directions and weights: key (upper left, front), fill (right), top
    (_unit([-0.5, 0.6, 0.62]), 0.78),
    (_unit([0.8, -0.1, 0.5]), 0.22),
    (_unit([0.0, 1.0, 0.2]), 0.12),
)


def mesh_faces(v, f, R, to_mm, base_rgb, vn=None, ambient=0.34, spec=0.10, shininess=24):
    """Shade one opaque mesh (back faces culled). Returns (polys [N,3,2] in mm, rgb [N,3], depth [N])."""
    pc = v @ R.T
    fn = face_normals(v, f) @ R.T
    if vn is not None:
        n = vn[f].mean(1)
        n = (n / (np.linalg.norm(n, axis=1, keepdims=True) + 1e-12)) @ R.T
    else:
        n = fn
    keep = fn[:, 2] > 0.0
    f, n = f[keep], n[keep]
    base = np.asarray(base_rgb, float)
    base = base[keep] if base.ndim == 2 else np.broadcast_to(base, (len(f), 3))
    V = np.array([0, 0, 1.0])
    dif = np.zeros(len(f))
    sp = np.zeros(len(f))
    for L, w in LIGHTS:
        dif += w * np.clip(n @ L, 0, 1)
        sp += w * np.clip(n @ _unit(L + V), 0, 1) ** shininess
    col = np.clip(base * (ambient + dif)[:, None] + spec * sp[:, None], 0, 1)
    return to_mm(pc[:, :2])[f], col, pc[f, 2].mean(1)


def draw_faces(ax, parts, z=2, lw=0.12):
    """Painter's sort over several mesh_faces() outputs; one rasterized PolyCollection."""
    P = np.concatenate([p[0] for p in parts])
    C = np.concatenate([p[1] for p in parts])
    D = np.concatenate([p[2] for p in parts])
    o = np.argsort(D)
    pc = PolyCollection(P[o], facecolors=C[o], edgecolors=C[o], linewidths=lw, zorder=z, antialiaseds=False)
    pc.set_rasterized(True)
    ax.add_collection(pc)
    return pc


def ellipsoid_outline(R, c, radii, s, off, n=360):
    """Screen silhouette of an ellipsoid under the orthographic camera, as an (n, 2) polyline in mm."""
    M = np.diag(radii)                                   # x = c + M u, |u| = 1
    A = R[:2] @ M                                        # screen = A u (+ const)
    U, S, Vt = np.linalg.svd(A)
    t = np.linspace(0, 2 * np.pi, n)
    pts = (U @ np.diag(S) @ np.c_[np.cos(t), np.sin(t)].T).T
    return s * (pts + (R[:2] @ c)) + off


def load_brain(cache=None):
    """fsaverage5 pial surfaces (both hemispheres) with per-vertex sulcal depth, outward-oriented faces."""
    cache = cache or os.path.join(os.path.expanduser("~"), ".cache", "s2i", "fs5_pial.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        return {h: (d[f"v_{h}"], d[f"f_{h}"], d[f"s_{h}"]) for h in ("left", "right")}
    from nilearn import datasets, surface
    fs = datasets.fetch_surf_fsaverage("fsaverage5")
    out, save = {}, {}
    for h in ("left", "right"):
        m = surface.load_surf_mesh(fs[f"pial_{h}"])
        v = np.asarray(m.coordinates if hasattr(m, "coordinates") else m[0], float)
        f = np.asarray(m.faces if hasattr(m, "faces") else m[1], int)
        s = np.asarray(surface.load_surf_data(fs[f"sulc_{h}"]), float)
        if ((v[f].mean(1) - v.mean(0)) * face_normals(v, f)).sum(1).mean() < 0:
            f = f[:, ::-1].copy()
        out[h] = (v, f, s)
        save.update({f"v_{h}": v, f"f_{h}": f, f"s_{h}": s})
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    np.savez_compressed(cache, **save)
    return out


def lateral_2d(n=1100, sigma=1.5, cache=None):
    """Flat lateral view of the right hemisphere (anterior to the right), for 2D vector drawing.

    The hemisphere is rasterized once off-screen (front faces, painter's sort, grey = sulcal depth); returns
    pixel-centre coordinates in brain mm (xs, ys), a smoothed inside-mask M and smoothed sulcal field T (0..1,
    > 0.5 = sulcus). Contour M at 0.5 for the outline and T at 0.5 for the gyral pattern.
    """
    cache = cache or os.path.join(os.path.expanduser("~"), ".cache", "s2i", f"fs5_lat2d_{n}_{sigma:g}.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        return d["xs"], d["ys"], d["M"], d["T"]
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    from matplotlib.figure import Figure
    from scipy.ndimage import gaussian_filter
    v, f, su = load_brain()["right"]
    R = camera(0.0, 0.0)                                  # viewer at +x; screen right = anterior
    vn = vertex_normals(v, f)
    v, f, vn, su = subdivide(v, f, vn, su)
    pc = v @ R.T
    f = f[(face_normals(v, f) @ R.T)[:, 2] > 0]
    P, D = pc[:, :2][f], pc[f, 2].mean(1)
    o = np.argsort(D)
    t = 1 / (1 + np.exp(-su[f].mean(1) / 0.5))
    lo, hi = pc[:, :2].min(0) - 4, pc[:, :2].max(0) + 4
    h = int(round(n * (hi[1] - lo[1]) / (hi[0] - lo[0])))
    fig = Figure(figsize=(n / 100, h / 100), dpi=100)
    FigureCanvasAgg(fig)
    fig.patch.set_facecolor((1.0, 0.0, 0.0))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(lo[0], hi[0])
    ax.set_ylim(lo[1], hi[1])
    ax.axis("off")
    c = np.c_[t, t, t][o]
    ax.add_collection(PolyCollection(P[o], facecolors=c, edgecolors=c, linewidths=0.4, antialiaseds=False))
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[::-1, :, :3].astype(float) / 255
    inside = ((img[:, :, 0] - img[:, :, 1]) < 0.5).astype(float)
    M = gaussian_filter(inside, sigma)
    T = gaussian_filter(img[:, :, 1] * inside, sigma * 1.1) / (gaussian_filter(inside, sigma * 1.1) + 1e-6)
    xs = np.linspace(lo[0], hi[0], img.shape[1])
    ys = np.linspace(lo[1], hi[1], img.shape[0])
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    np.savez_compressed(cache, xs=xs, ys=ys, M=M, T=T)
    return xs, ys, M, T

In [ ]:
%%writefile {CODE_DIR}/s2i/paperfigs.py
"""
Paper figures, version 2 (Phase 10 rework, 12 Oct 2026). Style: s2i/figstyle.py; plan: figure_plan.md.

Each figure reads the saved result files only and writes <name>.{pdf,svg,png} plus <name>_numbers.json, which lists
every number printed on the figure, so that it can be checked against the manuscript.

  fig2_main   a per-patient r: lagged ridge -> fixed untrained bank -> final model (primary level, strict)
              b all models incl. deep baselines (secondary level, session-wise normalisation, same trials)
              c calibration efficiency (primary level, strict; 5 / 10 / 20 % calibration on the same test trials)

Usage
  python -m s2i.paperfigs --runs RUNS_DIR --out FIG_DIR [--only fig2]
"""
import argparse
import glob
import json
import os

import numpy as np

from s2i import figstyle as st
from s2i.figstyle import (ACCENT, BASE, EXT, FIXED, INK, INK2, INK3, MM, OURS, OURS_SOFT, PATIENT, RULE, W2,
                          WASH, plt)

STRICT = "_qc_none"                      # primary level (no additional automatic QC)

# hand-tuned label offsets (data units unless noted); edit after looking at the real-data render
NUDGE = {
    "fig1": {"brain": (34.0, 34.0), "brain_w": 58.0},
    "fig2": {"a_base_lab": (-0.16, 0.0), "a_fixed_lab": (0.0, 0.012), "a_final_lab": (0.13, 0.0),
             "c_note": (0.22, 0.026), "c_dx": 0.07},
}


# ----------------------------------------------------------------------------- loading
def load_dir(path, prefix):
    out = {}
    for f in sorted(glob.glob(os.path.join(path, f"{prefix}sub-*.json"))):
        d = json.load(open(f))
        out[d.get("subject", os.path.basename(f)[len(prefix):-5])] = d
    return out


def load_strict(main_run, name="readout_combo5_strict"):
    """Primary-level strict readouts with every calibration fraction in one `fracs` dict: the 20 % run holds frac
    "0.2"; the 10 % and 5 % runs are separate folders (suffix _f10 / _f05, frac keys "0.1" / "0.05")."""
    D = load_dir(os.path.join(main_run, name + STRICT), "readout_")
    for suf in ("_f10", "_f05"):
        for s, d in load_dir(os.path.join(main_run, name + STRICT + suf), "readout_").items():
            if s in D:
                for f, v in d.get("fracs", {}).items():
                    D[s]["fracs"].setdefault(f, v)
    return D


def model_r(D, model, frac="0.2", key="r"):
    return {s: d["fracs"][frac]["models"][model][key] for s, d in D.items()
            if frac in d.get("fracs", {}) and model in d["fracs"][frac]["models"]}


def paired(*dicts):
    subs = sorted(set.intersection(*[set(d) for d in dicts]))
    return subs, [np.array([d[s] for s in subs], float) for d in dicts]


def find_leadformer(runs):
    for run in sorted(glob.glob(os.path.join(runs, "*"))):
        cfgp = os.path.join(run, "config.json")
        if not os.path.exists(cfgp):
            continue
        cfg = json.load(open(cfgp))
        if cfg.get("target") != "bip" or cfg.get("eeg_ref") != "car" or "calib_frac" not in cfg:
            continue
        out = {}
        for f in glob.glob(os.path.join(run, "fold_sub-*.json")):
            fo = json.load(open(f))
            if "calibrated" in fo:
                out[fo["subject"]] = float(np.nanmean(fo["calibrated"]["r"]))
        if out:
            return out
    return {}


def paired_stats(a, b):
    d = np.asarray(a) - np.asarray(b)
    lo, hi = st.boot_ci(d)
    return {"mean_diff": float(d.mean()), "ci": [lo, hi], "wins": int((d > 0).sum()), "n": int(len(d))}


def dump_numbers(out, name, nums):
    os.makedirs(out, exist_ok=True)
    json.dump(nums, open(os.path.join(out, f"{name}_numbers.json"), "w"), indent=1)


# ----------------------------------------------------------------------------- figure 1 (schematic, no data)
# Every coordinate below is in millimetres on the printed page (the axes fill the figure, 1 data unit = 1 mm).
def _rbox(ax, x, y, w, h, fc, ec="none", lw=0.6, r=1.0, ls="-", z=1):
    from matplotlib.patches import FancyBboxPatch
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}", fc=fc, ec=ec, lw=lw,
                                ls=ls, zorder=z))


def _arr(ax, p, q, color=INK2, lw=0.55, ls="-", rad=0.0, head=4.5, z=3):
    from matplotlib.patches import FancyArrowPatch
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=head, color=color, lw=lw, linestyle=ls,
                                 connectionstyle=f"arc3,rad={rad}", shrinkA=0, shrinkB=0, zorder=z))


def _sig(n, seed, f=(3, 7, 13, 21)):
    t = np.linspace(0, 1, n)
    g = np.random.default_rng(seed)
    s = sum(g.uniform(0.35, 1) / (1 + 0.08 * fi) * np.sin(2 * np.pi * (fi + g.uniform(-1, 1)) * t + g.uniform(0, 6))
            for fi in f)
    return t, s / np.abs(s).max()


def _wave(ax, xc, yc, w, h, cyc, color, lw=0.7, ls="-", skew=0.0, shift=0.0, z=4):
    t = np.linspace(-1, 1, 200)
    env = np.exp(-((t - shift) * 2.3) ** 2) * (1 + skew * t)
    s = env * np.cos(np.pi * cyc * t + skew * 2.0)
    ax.plot(xc + w / 2 * t, yc + h / 2 * s / np.abs(s).max(), color=color, lw=lw, ls=ls, zorder=z,
            solid_capstyle="round")


def _vdots(ax, x, y, gap=1.0, r=0.22, color=INK3):
    for k in (-1, 0, 1):
        ax.add_patch(plt.Circle((x, y + k * gap), r, fc=color, ec="none", zorder=3))


def _ptitle(ax, x, y, letter, title, tag):
    ax.text(1.6, y, letter, fontsize=8, fontweight="bold", color=INK, va="baseline")
    ax.text(x, y, title, fontsize=7, color=INK, va="baseline")
    ax.text(x, y - 3.3, tag, fontsize=5.6, color=INK3, va="baseline")


def _ucurve(ax, x0, y0, x1, y1, ybot, r=2.2, color=ACCENT, lw=0.8):
    """Arrow from (x0, y0) down to ybot, across, and up into (x1, y1), with rounded corners."""
    from matplotlib.patches import FancyArrowPatch
    from matplotlib.path import Path
    sx = np.sign(x1 - x0)
    v = [(x0, y0), (x0, ybot + r), (x0, ybot), (x0 + sx * r, ybot), (x1 - sx * r, ybot), (x1, ybot), (x1, ybot + r),
         (x1, y1)]
    c = [Path.MOVETO, Path.LINETO, Path.CURVE3, Path.CURVE3, Path.LINETO, Path.CURVE3, Path.CURVE3, Path.LINETO]
    ax.add_patch(FancyArrowPatch(path=Path(v, c), arrowstyle="-|>", mutation_scale=5, color=color, lw=lw, zorder=5))


def fig1_overview(runs, out):
    H = 104.0
    fig = plt.figure(figsize=(W2, H * MM))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 180)
    ax.set_ylim(0, H)
    ax.axis("off")
    TEST = "#d6d1c7"

    # ================= a: deployment-faithful evaluation (one patient's recording, as a row of trials)
    _ptitle(ax, 7, 100.2, "a", "Evaluated the way it would be deployed",
            "one patient's recording as a row of trials · hollow = artefact annotated in the dataset, excluded")
    y0, h, slot, bw, sgap = 84.0, 5.0, 1.62, 1.12, 3.2
    sess = [17, 15, 16, 12]                                   # trials per session (schematic)
    ncal, artef = 12, {21, 34, 35, 49}                        # 20 % of 60 trials; dataset-annotated artefacts
    xs, x, k = [], 7.0, 0
    for si, n in enumerate(sess):
        x_s0 = x
        for _ in range(n):
            xs.append(x)
            x += slot
        ax.text((x_s0 + x - slot + bw) / 2, y0 - 2.6, f"session {si + 1}", fontsize=5.2, color=INK3, ha="center",
                va="top")
        x += sgap
    for i, xi in enumerate(xs):
        if i < ncal:
            ax.add_patch(plt.Rectangle((xi, y0), bw, h, fc=OURS, ec="none", zorder=2))
        elif i == ncal:                                       # one-trial guard gap
            ax.plot([xi + bw / 2] * 2, [y0 - 0.3, y0 - 2.6], color=INK3, lw=0.35)
            ax.text(xi + bw / 2 + 0.6, y0 - 2.6, "1-trial gap", fontsize=5.0, color=INK3, ha="left", va="top")
        elif i in artef:
            ax.add_patch(plt.Rectangle((xi + 0.1, y0 + 0.1), bw - 0.2, h - 0.2, fc="white", ec=RULE, lw=0.4, zorder=2))
        else:
            ax.add_patch(plt.Rectangle((xi, y0), bw, h, fc=TEST, ec="none", zorder=2))
    xc1, xt0, xt1 = xs[ncal - 1] + bw, xs[ncal + 1], xs[-1] + bw
    yb = y0 + h + 1.3
    for (a_, b_, col) in ((xs[0], xc1, OURS), (xt0, xt1, INK2)):
        ax.plot([a_, a_, b_, b_], [yb - 0.7, yb, yb, yb - 0.7], color=col, lw=0.45, solid_joinstyle="miter")
    ax.text(xs[0], yb + 1.1, "calibration: first 20 % of trials (≈ 6 min)", fontsize=5.8, color=OURS, ha="left",
            va="bottom", fontweight="bold")
    ax.text(xt1, yb + 1.1, "test: every later trial, same and later sessions", fontsize=5.8, color=INK2, ha="right",
            va="bottom")
    ax.text(xt1 + 1.4, y0 + h / 2, "time", fontsize=5.2, color=INK3, va="center")
    _arr(ax, (xt1 + 6.6, y0 + h / 2), (xt1 + 9.6, y0 + h / 2), color=INK3, lw=0.45, head=3.5)
    # rules (numbered, hand-wrapped)
    rules = [("bipolar SEEG targets, common-average scalp EEG",), ("normalisation and hyper-parameters from",
             "calibration trials only"), ("no automatic artefact rejection beyond", "the dataset's annotations"),
             ("every contact and every patient reported",), ("statistics over patients (n = 15)",),
             ("two main hypotheses pre-specified and tested", "on an independent cohort (n = 12; Fig. 7)")]
    yy = 99.6
    for i, lines in enumerate(rules):
        col = EXT if i == len(rules) - 1 else INK2
        ax.text(130.0, yy, f"{i + 1}", fontsize=5.4, color=INK3 if i < len(rules) - 1 else EXT, va="baseline")
        for j, s in enumerate(lines):
            ax.text(133.0, yy - j * 2.45, s, fontsize=5.6, color=col, va="baseline")
        yy -= 2.45 * len(lines) + 0.55
    ax.plot([127.6, 127.6], [yy + 1.6, 101.6], color=RULE, lw=0.4)

    # ================= b: model for a new patient
    _ptitle(ax, 7, 73.6, "b", "Temporal filters are shared across patients; the spatial readout is not",
            "the model applied to a new patient")
    zb, zt = 34.4, 67.4
    _rbox(ax, 44.0, zb, 40.0, zt - zb, OURS_SOFT, r=1.6, z=0)
    _rbox(ax, 86.0, zb, 92.0, zt - zb, WASH, r=1.6, z=0)
    ax.text(45.6, zt - 2.3, "shared across patients", fontsize=5.6, color=OURS, fontweight="bold", va="center")
    ax.text(87.6, zt - 2.3, "fitted per patient, on calibration trials only", fontsize=5.6, color=INK2,
            fontweight="bold", va="center")
    # head, top view: scalp electrodes and one SEEG shaft
    cx, cy, R = 18.0, 52.0, 10.8
    th = np.linspace(0, 2 * np.pi, 300)
    ax.fill(cx + R * np.cos(th), cy + R * np.sin(th), color="white", zorder=1)
    ax.plot(cx + R * np.cos(th), cy + R * np.sin(th), color=INK2, lw=0.6, zorder=2)
    ax.plot([cx - 1.6, cx, cx + 1.6], [cy + R - 0.15, cy + R + 1.9, cy + R - 0.15], color=INK2, lw=0.6, zorder=2)
    for s in (-1, 1):
        e = np.linspace(-0.55, 0.55, 40)
        ax.plot(cx + s * (R + 0.9 * np.cos(e * 2.6)), cy + 2.6 * e * 1.4, color=INK2, lw=0.6, zorder=2)
    pos = {"Fp1": (-.30, .86), "Fp2": (.30, .86), "F7": (-.72, .47), "F3": (-.36, .42), "Fz": (0, .40),
           "F4": (.36, .42), "F8": (.72, .47), "T7": (-.88, 0), "C3": (-.44, 0), "Cz": (0, 0), "C4": (.44, 0),
           "T8": (.88, 0), "P7": (-.72, -.47), "P3": (-.36, -.42), "Pz": (0, -.40), "P4": (.36, -.42),
           "P8": (.72, -.47), "O1": (-.30, -.86), "O2": (.30, -.86)}
    E = {k: (cx + .82 * R * u, cy + .82 * R * v) for k, (u, v) in pos.items()}
    for k, (ex, ey) in E.items():
        ax.add_patch(plt.Circle((ex, ey), 0.62, fc=INK if k in ("F4", "C4", "P4") else "white", ec=INK, lw=0.4,
                                zorder=4))
    p0, p1 = np.array([2.6, 49.0]), np.array([16.8, 50.4])          # shaft from outside the head inwards
    ax.plot(*zip(p0, p1), color=INK3, lw=0.9, zorder=3, solid_capstyle="butt")
    for j, tt in enumerate(np.linspace(0.42, 0.97, 6)):
        q = p0 + tt * (p1 - p0)
        ax.add_patch(plt.Rectangle((q[0] - 0.5, q[1] - 0.5), 1.0, 1.0, angle=0, fc=OURS if j >= 4 else INK2,
                                   ec="none", zorder=4))
    ax.text(cx, zb + 4.0, "scalp EEG (6–19 channels)", fontsize=5.4, color=INK2, ha="center", va="center")
    ax.text(cx, zb + 1.6, "SEEG shaft, bipolar pairs", fontsize=5.4, color=OURS, ha="center", va="center")
    # scalp channels -> traces
    ty = {"F4": 59.0, "C4": 53.6, "P4": 44.6}
    for k, yv in ty.items():
        ax.plot([E[k][0] + 0.6, 31.0], [E[k][1], yv], color=INK3, lw=0.3, zorder=3)
        t, s = _sig(160, {"F4": 1, "C4": 2, "P4": 3}[k])
        ax.plot(31.4 + 10.6 * t, yv + 1.5 * s, color=INK, lw=0.5, zorder=3)
    _vdots(ax, 36.7, 49.4)
    # filter bank: a stack of identical cards = the same bank on every channel
    bx, by, bw_, bh = 48.4, 40.8, 22.0, 19.4
    for o in (2.6, 1.3):
        _rbox(ax, bx + o, by + o, bw_, bh, "white", OURS, lw=0.4, r=0.9, z=1)
    _rbox(ax, bx, by, bw_, bh, "white", OURS, lw=0.65, r=0.9, z=2)
    for j, (cyc, sk, sh) in enumerate(((1.4, 0.25, 0.08), (2.8, -0.2, -0.05), (5.0, 0.3, 0.04), (8.5, -0.15, 0.0))):
        yc = by + bh - 3.0 - j * 4.75
        _wave(ax, bx + bw_ / 2, yc, 17.0, 3.4, cyc, FIXED, lw=0.6, ls=(0, (1.6, 1.1)))
        _wave(ax, bx + bw_ / 2, yc, 17.0, 3.4, cyc * (1 + 0.06 * (j - 1.5)), OURS, lw=0.75, skew=sk, shift=sh)
    for k, yv in ty.items():
        _arr(ax, (42.6, yv), (bx - 0.5, by + 3 + (yv - 44.6) / 14.4 * (bh - 6)), color=INK2, lw=0.45, head=3.6)
    ax.text(bx, by - 2.0, "28 filters + 12 envelopes per channel", fontsize=5.1, color=INK2, va="center")
    ax.plot([bx, bx + 3.2], [by - 4.5] * 2, color=FIXED, lw=0.7, ls=(0, (1.6, 1.1)))
    ax.text(bx + 4.0, by - 4.5, "Morlet init", fontsize=5.1, color=INK3, va="center")
    ax.plot([bx + 15.4, bx + 18.6], [by - 4.5] * 2, color=OURS, lw=0.8)
    ax.text(bx + 19.4, by - 4.5, "meta-learned", fontsize=5.1, color=INK3, va="center")
    # features
    for o in range(4):
        ax.add_patch(plt.Rectangle((75.0 + o * 0.9, 46.4 + o * 0.9), 4.6, 8.4, fc="white", ec=OURS, lw=0.45,
                                   zorder=2 + o))
    ax.text(79.0, 44.5, "C × 40", fontsize=5.4, color=OURS, ha="center", va="center")
    _arr(ax, (bx + bw_ + 3.2, 51.5), (74.6, 51.5), color=INK2, lw=0.45, head=3.6)
    # ridge readout (matrix)
    g = np.random.default_rng(7)
    Wm = np.tanh(g.normal(0, 0.9, (9, 6)) + np.outer(np.sin(np.arange(9) / 1.7), np.cos(np.arange(6) / 1.3)))
    mx, my, cs = 90.4, 42.6, 1.95
    for i in range(9):
        for j in range(6):
            v = Wm[i, j]
            ax.add_patch(plt.Rectangle((mx + j * cs, my + i * cs), cs, cs, fc=OURS if v > 0 else BASE,
                                       alpha=0.12 + 0.75 * abs(v), ec="white", lw=0.35, zorder=2))
    ax.text(mx, my - 2.0, "ridge readout W", fontsize=5.4, color=INK, va="center")
    ax.text(mx, my - 3.4, "penalty by\nblocked 5-fold r", fontsize=5.1, color=INK3, va="top", linespacing=1.15)
    _arr(ax, (82.4, 51.5), (mx - 0.6, 51.5), color=INK2, lw=0.45, head=3.6)
    # outputs: recorded vs predicted bipolar SEEG
    ox0, ox1 = 107.6, 131.0
    for j, yv in enumerate((58.4, 52.9, 44.6)):
        t, s = _sig(220, 20 + j, f=(2, 5, 9, 16))
        p = np.convolve(s, np.ones(9) / 9, mode="same") * 0.78 + 0.1 * _sig(220, 40 + j, f=(11, 23))[1]
        ax.plot(ox0 + (ox1 - ox0) * t, yv + 1.7 * s, color=TEST, lw=1.25, zorder=2, solid_capstyle="round")
        ax.plot(ox0 + (ox1 - ox0) * t, yv + 1.7 * p, color=OURS, lw=0.6, zorder=3)
    _vdots(ax, (ox0 + ox1) / 2, 49.0)
    ax.plot([ox0, ox0 + 3.4], [62.4] * 2, color=TEST, lw=1.25)
    ax.text(ox0 + 4.1, 62.4, "recorded", fontsize=5.1, color=INK3, va="center")
    ax.plot([ox0 + 13.2, ox0 + 16.6], [62.4] * 2, color=OURS, lw=0.7)
    ax.text(ox0 + 17.3, 62.4, "predicted", fontsize=5.1, color=INK3, va="center")
    _arr(ax, (mx + 6 * cs + 1.0, 51.5), (ox0 - 1.2, 51.5), color=INK2, lw=0.45, head=3.6)
    ax.text(ox0 - 1.0, my - 2.0, "bipolar SEEG, every contact", fontsize=5.4, color=INK, va="center")
    ax.text(ox0 - 1.0, my - 4.5, "final: 5 encoders averaged, mixed with ridge", fontsize=5.1, color=INK3,
            va="center")
    # contact ranking from calibration cross-validation
    rx = 147.0
    vals = np.array([.43, .39, .36, .31, .28, .26, .22, .19, .16, .12, .09, .05])
    for i, v in enumerate(vals):
        yv = 58.8 - i * 1.42
        ax.add_patch(plt.Rectangle((rx, yv - 0.48), v / vals[0] * 25.0, 0.96, fc=OURS if i < 3 else TEST, ec="none",
                                   zorder=2))
    ax.plot([rx - 0.9, rx - 1.6, rx - 1.6, rx - 0.9], [58.8 + 0.6, 58.8 + 0.6, 58.8 - 2 * 1.42 - 0.6,
            58.8 - 2 * 1.42 - 0.6], color=OURS, lw=0.45, solid_joinstyle="miter")
    ax.text(rx - 2.4, 58.8 - 1.42, "top\nquarter", fontsize=5.0, color=OURS, ha="right", va="center",
            linespacing=1.05)
    ax.text(rx, 62.4, "contacts ranked by calibration r", fontsize=5.1, color=INK3, va="center")
    ax.text(rx, my - 2.0, "which contacts to trust", fontsize=5.4, color=INK, va="center")
    ax.text(rx, my - 4.5, "before any test data", fontsize=5.1, color=INK3, va="center")
    _arr(ax, (ox1 + 1.2, 51.5), (rx - 7.2, 51.5), color=INK3, lw=0.45, ls=(0, (2, 1.5)), head=3.6)

    # ================= c: episodic meta-training
    _ptitle(ax, 7, 30.8, "c", "The filters are meta-learned through the ridge readout",
            "episodic training on the other patients (leave-one-patient-out)")
    # patient pool
    gx, gy = [7.0, 10.0, 13.0, 16.0, 19.0], [21.6, 18.1, 14.6]
    role = {(1, 2): "test", (2, 0): "stop", (0, 3): "stop"}
    for r_, yv in enumerate(gy):
        for c_, xv in enumerate(gx):
            k = role.get((r_, c_), "train")
            if k == "test":
                ax.add_patch(plt.Circle((xv, yv), 1.05, fc="white", ec=INK, lw=0.7, zorder=3))
            elif k == "stop":
                ax.add_patch(plt.Circle((xv, yv), 1.0, fc="white", ec=BASE, lw=0.45, ls=(0, (1.2, 0.8)), zorder=3))
            else:
                ax.add_patch(plt.Circle((xv, yv), 1.0, fc=BASE, ec="none", zorder=3))
    for j, (k, s) in enumerate((("test", "held-out test patient"), ("stop", "2 for early stopping"),
                                ("train", "12 for training"))):
        yv = 10.0 - j * 2.7
        if k == "test":
            ax.add_patch(plt.Circle((7.0, yv), 0.8, fc="white", ec=INK, lw=0.6))
        elif k == "stop":
            ax.add_patch(plt.Circle((7.0, yv), 0.75, fc="white", ec=BASE, lw=0.45, ls=(0, (1.2, 0.8))))
        else:
            ax.add_patch(plt.Circle((7.0, yv), 0.75, fc=BASE, ec="none"))
        ax.text(8.8, yv, s, fontsize=5.2, color=INK2, va="center")
    _arr(ax, (21.4, 18.1), (28.6, 18.1), color=INK2, lw=0.45, head=3.6)
    ax.text(25.0, 19.3, "sample 2", fontsize=5.0, color=INK3, ha="center", va="bottom")
    # episode: support / gap / query
    ex0, es, ebw, ey, eh = 30.0, 1.5, 1.05, 15.6, 5.0
    for i in range(20):
        xi = ex0 + i * es
        if i < 11:
            ax.add_patch(plt.Rectangle((xi, ey), ebw, eh, fc=OURS_SOFT, ec=OURS, lw=0.35))
        elif i > 11:
            ax.add_patch(plt.Rectangle((xi, ey), ebw, eh, fc="white", ec=OURS, lw=0.35))
    ax.text(ex0, ey + eh + 1.0, "support\n≈ 4 min", fontsize=5.2, color=OURS, va="bottom", linespacing=1.1)
    ax.text(ex0 + 20 * es - (es - ebw), ey + eh + 1.0, "query\n8 trials", fontsize=5.2, color=OURS, va="bottom",
            ha="right", linespacing=1.1)
    ax.text(ex0 + 11 * es + ebw / 2, ey - 1.4, "gap", fontsize=4.8, color=INK3, ha="center", va="top")
    # encoder chip -> ridge -> loss
    cx0, cw, cy0, ch = 66.0, 13.0, 13.6, 9.0
    _rbox(ax, cx0, cy0, cw, ch, "white", OURS, lw=0.65, r=0.9, z=2)
    for j, cyc in enumerate((1.6, 3.4, 7.0)):
        _wave(ax, cx0 + cw / 2, cy0 + ch - 1.8 - j * 2.7, 10.0, 1.9, cyc, OURS, lw=0.6, skew=0.15 * (j - 1))
    ax.text(cx0 + cw / 2, cy0 + ch + 1.2, "filter bank", fontsize=5.2, color=OURS, ha="center", va="bottom")
    _arr(ax, (ex0 + 20 * es + 0.6, 18.1), (cx0 - 0.5, 18.1), color=INK2, lw=0.45, head=3.6)
    rx0, rw = 85.0, 30.0
    _rbox(ax, rx0, cy0, rw, ch, "white", BASE, lw=0.55, r=0.9, z=2)
    ax.text(rx0 + rw / 2, cy0 + ch - 2.5, "closed-form ridge on support", fontsize=5.6, color=INK, ha="center",
            va="center")
    ax.text(rx0 + rw / 2, cy0 + 2.8, r"$W = (X^{\mathrm{T}}X + \lambda I)^{-1}X^{\mathrm{T}}Y$", fontsize=5.6, color=INK2,
            ha="center", va="center")
    _arr(ax, (cx0 + cw + 0.6, 18.1), (rx0 - 0.5, 18.1), color=INK2, lw=0.45, head=3.6)
    lx0, lw_ = 121.0, 22.0
    _rbox(ax, lx0, cy0, lw_, ch, "white", BASE, lw=0.55, r=0.9, z=2)
    ax.text(lx0 + lw_ / 2, cy0 + ch - 2.5, "predict query", fontsize=5.6, color=INK, ha="center", va="center")
    ax.text(lx0 + lw_ / 2, cy0 + 2.8, "loss = 1 − mean r", fontsize=5.6, color=INK2, ha="center", va="center")
    _arr(ax, (rx0 + rw + 0.6, 18.1), (lx0 - 0.5, 18.1), color=INK2, lw=0.45, head=3.6)
    # the accent: gradient back through the closed-form solution into the filters (and lambda)
    _ucurve(ax, lx0 + lw_ / 2, cy0 - 0.3, cx0 + cw / 2, cy0 - 0.4, 6.4)
    _arr(ax, (rx0 + rw / 2 + 6.5, 6.4), (rx0 + rw / 2 + 6.5, cy0 - 0.4), color=ACCENT, lw=0.6, head=4.2, z=5)
    ax.text(rx0 + rw / 2 + 7.6, 9.6, "λ", fontsize=5.4, color=ACCENT, va="center")
    ax.text((cx0 + lx0 + lw_) / 2, 4.6, "gradient back-propagated through the ridge solution", fontsize=5.6,
            color=ACCENT, ha="center", va="top")
    # take-home
    for j, s in enumerate(("The readout is refitted in every", "episode, so only what transfers",
                           "across patients is learned:", "the temporal filters.")):
        ax.text(148.0, 22.2 - j * 2.6, s, fontsize=5.6, color=INK if j == 3 else INK2, va="baseline",
                fontweight="bold" if j == 3 else "normal")
    ax.plot([146.2, 146.2], [22.2 - 3 * 2.6 - 0.8, 22.2 + 2.0], color=RULE, lw=0.4)

    st.save(fig, out, "fig1_overview")
    dump_numbers(out, "fig1_overview", {"calibration_label_min": "≈ 6 (manuscript: about 6 min; primary 5.7)",
                                         "support_s": 240, "query_trials": 8, "train_patients": 12,
                                         "early_stop_patients": 2, "features_per_channel": 40,
                                         "external_n": 12, "n_patients": 15})


# ----------------------------------------------------------------------------- figure 1, v3 (white scene)
# One left-to-right scene, few words; detail goes to the caption. 1 data unit = 1 mm on the page.
def _trace(n, seed, k=30, alpha=1.0):
    g = np.random.default_rng(seed)
    t = np.linspace(0, 1, n)
    y = sum(np.sin(2 * np.pi * f * 1.6 * t + g.uniform(0, 6.3)) / f ** (0.5 * alpha) for f in range(1, k))
    return y / np.abs(y).max()


FIG1_CH = ("#1a9a8b", "#7051a6", "#3f88c5", "#d39a27")   # four scalp channels (teal, violet, azure, ochre)
BRAIN_GY, BRAIN_SU, BRAIN_LN = "#f7ebe5", "#ead2c8", "#d6b2a5"
PATIENT_DK = "#bdb6aa"                               # recorded SEEG trace (a touch darker than PATIENT)


def _wt_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("wt", ["#d39a27", "#fbf9f5", OURS])


def _seq_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("q", ["#eef4fa", "#6aa6d6", OURS])


def _brain2d(ax, cx, cy, w, z=2.0, outline_only=False, fc=None, ec=INK2, lw=0.6, ls="-"):
    """Flat lateral right hemisphere (fsaverage5, anterior right) centred at (cx, cy), w mm wide.
    Returns (outline polyline in mm, bbox (x0, y0, x1, y1))."""
    import contourpy
    from matplotlib.patches import Polygon
    from s2i import render3d as r3
    xs, ys, M, T = r3.lateral_2d()
    raw = max(contourpy.contour_generator(xs, ys, M).lines(0.5), key=len)
    x0, x1, y0, y1 = raw[:, 0].min(), raw[:, 0].max(), raw[:, 1].min(), raw[:, 1].max()
    s = w / (x1 - x0)
    X, Y = cx + s * (xs - (x0 + x1) / 2), cy + s * (ys - (y0 + y1) / 2)
    out = np.c_[cx + s * (raw[:, 0] - (x0 + x1) / 2), cy + s * (raw[:, 1] - (y0 + y1) / 2)]
    bb = (out[:, 0].min(), out[:, 1].min(), out[:, 0].max(), out[:, 1].max())
    if outline_only:
        ax.add_patch(Polygon(out, closed=True, fc=fc or "none", ec=ec, lw=lw, ls=ls, zorder=z))
        return out, bb
    poly = Polygon(out, closed=True, fc=BRAIN_GY, ec="none", zorder=z)
    ax.add_patch(poly)
    cs = ax.contourf(X, Y, np.where(M > 0.2, T, 0.0), levels=[0.45, 2.0], colors=[BRAIN_SU], zorder=z + 0.1)
    cs.set_clip_path(poly)
    cl = ax.contour(X, Y, np.where(M > 0.2, T, 0.0), levels=[0.45], colors=[BRAIN_LN], linewidths=0.3,
                    zorder=z + 0.2)
    cl.set_clip_path(poly)
    ax.plot(out[:, 0], out[:, 1], color=ec, lw=lw, zorder=z + 0.3)
    return out, bb


def _head2d(ax, cx, cy, w, q, pad=(2.5, 7.0)):
    """Flat head: brain, scalp arc with electrodes (4 coloured = shown channels) and one SEEG shaft whose
    contacts carry the same colours as the SEEG panel. Sizes scale with w."""
    out, (x0, y0, x1, y1) = _brain2d(ax, cx, cy, w, lw=0.7)
    bw_, bh_ = x1 - x0, y1 - y0
    k = w / 40.0
    hc = np.array([cx, cy + 0.10 * bh_])
    hr = np.array([bw_ / 2 + pad[0], bh_ / 2 + pad[1]])
    t = np.radians(np.linspace(-22, 202, 300))
    ax.plot(hc[0] + hr[0] * np.cos(t), hc[1] + hr[1] * np.sin(t), color=INK3, lw=0.6, zorder=3)
    on = lambda a: hc + hr * np.array([np.cos(np.radians(a)), np.sin(np.radians(a))])      # noqa: E731
    for a in (178, 160, 142, 124, 106):
        ax.add_patch(plt.Circle(on(a), 0.55 * k, fc="white", ec=INK3, lw=0.45, zorder=4))
    for a, c in zip((84, 62, 40, 18), FIG1_CH):
        ax.add_patch(plt.Circle(on(a), 0.85 * k, fc=c, ec="white", lw=0.45, zorder=4.2))
    # SEEG shaft: in through the vertex, tip in the temporal lobe
    pe = on(114) + np.array([0.0, 1.8 * k])
    pt = np.array([x0 + 0.60 * bw_, y0 + 0.24 * bh_])
    ax.plot([pe[0], pt[0]], [pe[1], pt[1]], color=INK, lw=0.85, zorder=5, solid_capstyle="butt")
    u = (pe - pt) / np.linalg.norm(pe - pt)
    cm = _seq_cmap()
    for j, qk in enumerate(q):                            # j = 0 shallowest ... last = tip
        p = pt + u * 1.45 * k * (len(q) - 1 - j)
        ax.add_patch(plt.Circle(p, 0.55 * k, fc=cm(qk), ec=INK, lw=0.35, zorder=5.1))


def fig1_scene(runs, out):
    """Two-tier scene: the patient's head spans the full height on the left; the model runs along the top
    (scalp EEG -> shared filters -> readout) and drops into the predicted SEEG along the bottom."""
    H = 74.0
    yh = 70.0                                             # baseline of the top-row labels
    fig = plt.figure(figsize=(W2, H * MM))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 180)
    ax.set_ylim(0, H)
    ax.axis("off")
    N = NUDGE["fig1"]
    fs, fs2 = 6.5, 5.6
    q = np.array([0.82, 0.35, 0.58, 0.95, 0.70, 0.22, 0.45, 0.30])     # schematic per-contact accuracy
    mid = 57.25                                           # top-row flow line

    # ---- new patient: big flat head on the left, full height
    _head2d(ax, *N["brain"], w=N["brain_w"], q=q)
    ax.text(N["brain"][0], 3.6, "new patient", ha="center", fontsize=fs, color=INK)

    # ---- scalp EEG: four channels in the electrode colours, gathered by a bracket into the filters
    tx0, tx1 = 69.5, 94.0
    ys = [mid + 8.25, mid + 2.75, mid - 2.75, mid - 8.25]
    for j, (y, c) in enumerate(zip(ys, FIG1_CH)):
        x = np.linspace(tx0, tx1, 200)
        ax.plot(x, y + 1.6 * _trace(200, 10 + j), color=c, lw=0.65, zorder=3)
    xb_ = tx1 + 1.6
    ax.plot([xb_ - 0.8, xb_, xb_, xb_ - 0.8], [ys[0] + 1.9, ys[0] + 1.9, ys[-1] - 1.9, ys[-1] - 1.9],
            color=INK3, lw=0.5, zorder=3, solid_joinstyle="miter")
    ax.text((tx0 + tx1) / 2, yh, "scalp EEG", ha="center", fontsize=fs, color=INK)

    # ---- the shared, meta-learned filter bank (the idea of the paper) + the 14 patients behind it
    bx, bw, bh = 100.0, 34.0, 19.0
    by = mid - bh / 2
    _arr(ax, (xb_, mid), (bx - 0.6, mid), color=INK2, lw=0.55, head=4)
    for d in (1.8, 0.9):
        _rbox(ax, bx + d, by + d, bw, bh, "white", ec=OURS, lw=0.4, r=1.2, z=2)
    _rbox(ax, bx, by, bw, bh, OURS_SOFT, ec=OURS, lw=0.75, r=1.2, z=3)
    for j, (cyc, sk) in enumerate(((2.2, 0.0), (3.6, 0.25), (5.6, -0.2))):
        yc = mid + 5.6 - 5.6 * j
        _wave(ax, bx + bw / 2, yc, bw - 8, 4.4, cyc, FIXED, lw=0.6, ls=(0, (1.6, 1.1)), z=4)
        _wave(ax, bx + bw / 2, yc, bw - 8, 4.4, cyc * 1.12, OURS, lw=1.0, skew=sk, shift=0.06 * (j - 1), z=5)
    ax.text(bx + bw / 2, yh, "shared temporal filters", ha="center", fontsize=fs, color=OURS, fontweight="bold")
    for i in range(14):                                   # 12 training (filled), 2 early stopping (dashed)
        es = i in (5, 10)
        _brain2d(ax, bx + 1.25 + i * (bw - 2.5) / 13, by - 2.7, 2.25, z=3, outline_only=True,
                 fc="white" if es else "#a9c1da", ec=INK3 if es else OURS, lw=0.3, ls=(0, (1.0, 0.7)) if es else "-")
    ax.text(bx + bw / 2 + 1.5, by - 6.6, "learned on 14 other patients", ha="center", fontsize=fs2, color=INK2)

    # ---- per-patient readout (fitted on the first 20 % of trials)
    rx, rw, rh = 146.0, 16.0, 22.0
    ry = mid - rh / 2
    g = np.random.default_rng(4)
    Wm = np.tanh(g.normal(0, 1, (9, 6)) * 0.9)
    ax.imshow(Wm, extent=[rx, rx + rw, ry, ry + rh], cmap=_wt_cmap(), vmin=-1, vmax=1, zorder=3,
              interpolation="nearest", aspect="auto")
    ax.add_patch(plt.Rectangle((rx, ry), rw, rh, fc="none", ec=INK2, lw=0.5, zorder=4))
    ax.text(rx + rw / 2 + 2.0, yh, "per-patient readout", ha="center", fontsize=fs, color=INK)
    ax.text(rx + rw + 1.6, mid, "first\n20 % of\ntrials", ha="left", va="center", fontsize=fs2, color=OURS,
            linespacing=1.1)

    # ---- forward pass (grey) and, just under it, the one accent: the gradient back through the ridge
    x_a, x_b = bx + bw + 3.0, rx - 1.0
    yg = mid - 3.8
    _arr(ax, (x_a, mid), (x_b, mid), color=INK2, lw=0.55, head=4)
    _arr(ax, (x_b, yg), (x_a, yg), color=ACCENT, lw=1.05, head=5.5, z=6)
    ax.text((x_a + x_b) / 2, yg - 1.5, "gradient\nthrough\nridge", ha="center", va="top", fontsize=fs2,
            color=ACCENT, linespacing=1.05)

    # ---- bipolar SEEG along the bottom, contacts coloured by how well they are predicted (as in the brain)
    shx, y0, y1 = 73.0, 3.5, 37.5
    cy_ = np.linspace(y1 - 2.0, y0 + 2.0, len(q))
    cm = _seq_cmap()
    ax.plot([shx, shx], [y0, y1], color=INK, lw=0.85, zorder=3, solid_capstyle="butt")
    for j, (yc, qk) in enumerate(zip(cy_, q)):
        ax.add_patch(plt.Rectangle((shx - 1.25, yc - 1.05), 2.5, 2.1, fc=cm(qk), ec=INK, lw=0.4, zorder=4))
        xx = np.linspace(shx + 3.2, 178.5, 320)
        rec = _trace(320, 40 + j, k=40, alpha=1.3)
        pred = qk * rec + (1 - qk) * _trace(320, 80 + j, k=40, alpha=1.3)
        pred = pred / np.abs(pred).max()
        ax.plot(xx, yc + 1.5 * rec, color=PATIENT_DK, lw=1.0, zorder=3, solid_capstyle="round")
        ax.plot(xx, yc + 1.5 * pred, color=OURS, lw=0.6, alpha=0.35 + 0.65 * qk, zorder=4)
    _arr(ax, (rx + rw / 2, ry - 0.8), (rx + rw / 2, y1 + 1.2), color=INK2, lw=0.55, head=4)
    ax.text(shx - 3.2, (y0 + y1) / 2, "bipolar SEEG", ha="center", va="center", rotation=90, fontsize=fs,
            color=INK)
    ly = 41.0
    ax.plot([shx - 3.0, shx + 0.0], [ly] * 2, color=OURS, lw=0.6)
    ax.text(shx + 0.6, ly, "predicted", fontsize=fs2, color=INK2, va="center")
    ax.plot([shx + 12.2, shx + 15.2], [ly] * 2, color=PATIENT_DK, lw=1.0)
    ax.text(shx + 15.8, ly, "recorded", fontsize=fs2, color=INK2, va="center")

    st.save(fig, out, "fig1_overview")
    dump_numbers(out, "fig1_overview", {"schematic": True, "calibration_fraction": 0.2, "other_patients": 14,
                                         "early_stop_patients": 2})


def st_mix(c0, c1, t):
    from s2i.render3d import mix
    return tuple(mix(c0, c1, t))


def _div_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("w", [INK3, "#f4f2ee", OURS])


# ----------------------------------------------------------------------------- figure 2
def fig2_main(runs, out):
    N = NUDGE["fig2"]
    main_run, noenv = os.path.join(runs, "shared_bip_car"), os.path.join(runs, "shared_no_env")
    S5 = load_strict(main_run)
    SF = load_dir(os.path.join(noenv, "readout_fixed_strict" + STRICT), "readout_")
    if not S5 or not SF:
        raise FileNotFoundError(2, "missing", os.path.join(main_run, "readout_combo5_strict" + STRICT))
    subs, (base, fixed, final) = paired(model_r(S5, "base"), model_r(SF, "combo_nosess"), model_r(S5, "combo_nosess"))
    mins = float(np.mean([d["fracs"]["0.2"].get("minutes", np.nan) for d in S5.values()]))
    nums = {"a": {"n": len(subs), "mean_base": base.mean(), "mean_fixed": fixed.mean(), "mean_final": final.mean(),
                  "final_minus_base": paired_stats(final, base), "minutes_20": mins}}

    fig = plt.figure(figsize=(W2, 80 * MM))
    gs = fig.add_gridspec(2, 2, width_ratios=[0.78, 1.0], height_ratios=[1.0, 1.0], wspace=0.34, hspace=0.78,
                          left=0.075, right=0.975, top=0.875, bottom=0.115)

    # ---------------- a: paired per-patient lines (tall, left)
    ax = fig.add_subplot(gs[:, 0])
    xs = np.array([0.0, 1.0, 2.0])
    for b, f, c in zip(base, fixed, final):
        ax.plot(xs, [b, f, c], color=PATIENT, lw=0.45, zorder=1)
        ax.scatter(xs, [b, f, c], s=2.2, color=PATIENT, zorder=2, linewidths=0)
    m = np.array([base.mean(), fixed.mean(), final.mean()])
    ax.plot(xs, m, color=OURS, lw=1.3, zorder=3)
    for x, v, fc in zip(xs, m, (BASE, FIXED, OURS)):
        ax.scatter([x], [v], s=17, color=fc, edgecolors="white", linewidths=0.6, zorder=4)
    # mean labels next to the points
    dx, dy = N["a_base_lab"]
    ax.text(xs[0] + dx, m[0] + dy, st.fmt(m[0]), ha="right", va="center", fontsize=6.5, color=BASE, fontweight="bold")
    dx, dy = N["a_fixed_lab"]
    ax.text(xs[1] + dx, m[1] + dy, st.fmt(m[1]), ha="center", va="bottom", fontsize=6.5, color=OURS)
    # the effect: bracket from the base level to the final level, right of the final point
    xb = xs[2] + 0.2
    ax.plot([xs[0], xb], [m[0], m[0]], color=RULE, lw=0.5, ls=(0, (2, 2)), zorder=0)
    ax.text(xs[2] + 0.11, m[2], st.fmt(m[2]), ha="left", va="bottom", fontsize=6.5, color=OURS, fontweight="bold")
    s_ = nums["a"]["final_minus_base"]
    st.bracket(ax, xb + 0.02, m[0], m[2] - 0.004, st.fmt(s_["mean_diff"], sign=True), color=ACCENT, w=0.07, pad=0.05)
    ax.text(xb + 0.14, (m[0] + m[2]) / 2 - 0.0125,
            f"[{st.fmt(s_['ci'][0])}, {st.fmt(s_['ci'][1])}]\n{s_['wins']}/{s_['n']} patients",
            fontsize=5.8, color=INK2, va="top", ha="left", linespacing=1.25, clip_on=False)
    ax.set_xticks(xs, ["lagged\nridge", "fixed\nuntrained bank", "final\nmodel"])
    ax.tick_params(axis="x", length=0, pad=3)
    ax.spines["bottom"].set_visible(False)
    ax.set_xlim(-0.42, 2.75)
    lo_, hi_ = min(base.min(), fixed.min(), final.min()), max(base.max(), fixed.max(), final.max())
    ax.set_ylim(lo_ - 0.012, hi_ + 0.008)
    ax.set_yticks(np.arange(np.ceil((lo_ - 0.012) * 20) / 20, hi_ + 0.008 + 1e-9, 0.05))
    ax.set_ylabel("test r, mean over contacts")
    st.range_frame(ax, x=False)
    st.headline(ax, "The final model is above ridge in every patient",
                tag=f"primary level · strict normalisation · {mins:.1f} min calibration")
    axa = ax

    # ---------------- b: all models (top right), session-wise normalisation
    T5 = load_dir(os.path.join(main_run, "readout_combo5"), "readout_")
    TF = load_dir(os.path.join(noenv, "readout_fixed"), "readout_")
    nfn = {s: d["models"]["nfn_mode"]["r"] for s, d in load_dir(os.path.join(runs, "nfn_baseline"), "nfn_").items()}
    rows = [("final model", model_r(T5, "combo"), OURS), (st.LBL_FIXED, model_r(TF, "combo"), FIXED),
            ("lagged ridge", model_r(T5, "base"), BASE), ("LeadFormer", find_leadformer(runs), BASE),
            ("NeuroFlowNet", nfn, BASE)]
    rows = [r for r in rows if r[1]]
    ax = fig.add_subplot(gs[0, 1])
    yy = np.arange(len(rows))[::-1].astype(float)
    ax.axhspan(yy[1] - 0.5, yy[0] + 0.5, color=WASH, lw=0, zorder=0)        # our two rows
    nums["b"] = {}
    for y, (lab, d, col) in zip(yy, rows):
        v = np.array(list(d.values()), float)
        off = st.beeswarm(v, size_data=0.0065, max_off=0.30)
        dot = col if col != BASE else "#b3aea5"
        ax.scatter(v, y + off, s=2.6, color=dot, linewidths=0, zorder=2, alpha=0.95)
        lo, hi = st.boot_ci(v)
        mc = OURS if col in (OURS, FIXED) else INK2
        ax.plot([lo, hi], [y, y], color=mc, lw=1.0, zorder=3)
        ax.scatter([v.mean()], [y], s=13, marker="D", color=col if col != BASE else INK2, edgecolors="white",
                   linewidths=0.5, zorder=4)
        ax.text(max(hi, v.max()) + 0.012, y, st.fmt(v.mean()), va="center", ha="left", fontsize=6.2, color=mc,
                fontweight="bold" if lab == "final model" else "normal")
        nums["b"][lab] = {"n": len(v), "mean": float(v.mean()), "ci": [lo, hi]}
    ax.set_yticks(yy, [r[0] for r in rows])
    for t, r in zip(ax.get_yticklabels(), rows):
        t.set_color(OURS if r[2] in (OURS, FIXED) else INK)
    st.bare_y(ax)
    ax.set_ylim(yy[-1] - 0.6, yy[0] + 0.6)
    ax.set_xlim(0, 0.42)
    ax.set_xticks([0, 0.1, 0.2, 0.3, 0.4])
    ax.set_xlabel("test r per patient")
    if "lagged ridge" in nums["b"]:
        st.ref_line(ax, nums["b"]["lagged ridge"]["mean"], axis="x")
    st.range_frame(ax, y=False)
    st.headline(ax, "Above every baseline on the same trials",
                tag="secondary level · session-wise normalisation · diamond = mean, bar = 95 % CI")
    axb = ax

    # ---------------- c: calibration efficiency (bottom right), primary level, strict
    ax = fig.add_subplot(gs[1, 1])
    fr = [f for f in ("0.05", "0.1", "0.2") if all(f in d["fracs"] for d in S5.values())]
    xm = np.array([np.mean([d["fracs"][f]["minutes"] for d in S5.values()]) for f in fr])
    Y = {k: np.array([[d["fracs"][f]["models"][k]["r"] for f in fr] for d in S5.values()]) for k in ("base", "combo_nosess")}
    nums["c"] = {"minutes": xm.tolist()}
    cdx = N["c_dx"]
    for k, col, lab, sh in (("base", BASE, "lagged ridge", -cdx), ("combo_nosess", OURS, "final model", cdx)):
        mu = Y[k].mean(0)
        ci = np.array([st.boot_ci(Y[k][:, j], 4000) for j in range(len(fr))])
        # CI as thin whiskers, nudged apart horizontally so the two models never sit on top of each other
        for x, (l, h) in zip(xm + sh, ci):
            ax.plot([x, x], [l, h], color=col, lw=0.55, alpha=0.75, solid_capstyle="butt", zorder=2)
        ax.plot(xm + sh, mu, color=col, lw=1.1, zorder=3)
        ax.scatter(xm + sh, mu, s=10, color=col, edgecolors="white", linewidths=0.5, zorder=4)
        ax.text(xm[-1] + 0.28, mu[-1], lab, color=col, fontsize=6.2, va="center", ha="left")
        nums["c"][lab] = mu.tolist()
        nums["c"][lab + "_ci"] = ci.tolist()
    note_top = -np.inf
    if "0.1" in fr and "0.2" in fr:
        j10, j20 = fr.index("0.1"), fr.index("0.2")
        yb20 = Y["base"][:, j20].mean()
        ax.plot([xm[0] - 0.25, xm[j20] - cdx], [yb20, yb20], color=RULE, lw=0.5, ls=(0, (2, 2)), zorder=0)
        s_ = paired_stats(Y["combo_nosess"][:, j10], Y["base"][:, j20])
        nums["c"]["final10_minus_base20"] = s_
        y10 = Y["combo_nosess"][:, j10].mean()
        x10 = xm[j10] + cdx
        ax.scatter([x10], [y10], s=48, facecolors="none", edgecolors=INK, linewidths=0.6, zorder=5)
        dx, dy = N["c_note"]
        note_top = y10 + dy + 0.027
        ax.annotate(f"final {xm[j10]:.1f} min − ridge {xm[j20]:.1f} min\n"
                    f"{st.fmt(s_['mean_diff'], sign=True)} [{st.fmt(s_['ci'][0])}, {st.fmt(s_['ci'][1])}], "
                    f"{s_['wins']}/{s_['n']}",
                    xy=(x10, y10), xytext=(x10 + dx, y10 + dy),
                    fontsize=5.8, color=INK, ha="left", va="bottom", linespacing=1.3,
                    arrowprops=dict(arrowstyle="-", color=INK, lw=0.45, shrinkA=0.5, shrinkB=3.2,
                                    connectionstyle="angle,angleA=0,angleB=90,rad=0"))
    ax.set_xticks(xm, [f"{v:.1f}" for v in xm])
    ax.set_xlim(xm[0] - 0.4, xm[-1] + 1.9)
    allci = np.concatenate([np.ravel(nums["c"][k]) for k in nums["c"] if k.endswith("_ci")])
    lo_, hi_ = allci.min(), allci.max()
    ax.set_ylim(lo_ - 0.006, max(hi_, note_top) + 0.004)
    ax.set_yticks(np.round(np.arange(np.ceil(lo_ * 25) / 25, hi_ + 1e-9, 0.04), 3))
    ax.set_xlabel("calibration (min)")
    ax.set_ylabel("test r")
    st.range_frame(ax)
    st.headline(ax, "Half the calibration already beats ridge",
                tag="primary level · strict · same test trials · ds004752 only (not replicated, Fig. 7)")
    axc = ax

    st.letters(fig, [(axa, "a", -1.5), (axb, "b", -1.5), (axc, "c", -1.5)])
    st.save(fig, out, "fig2_main")
    dump_numbers(out, "fig2_main", nums)


FIGS = {"fig1": fig1_scene, "fig2": fig2_main}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--only", nargs="*", default=None)
    ap.add_argument("--style", default="own", choices=["own", "sp"], help="sp = SciencePlots base style")
    a = ap.parse_args()
    st.setup(a.style)
    from s2i import paperfigs_b
    FIGS.update(paperfigs_b.FIGS)
    for k, fn in FIGS.items():
        if a.only and k not in a.only:
            continue
        print(k)
        try:
            fn(a.runs, a.out)
        except FileNotFoundError as e:
            print(f"  skipped: missing {e.filename}")
        except Exception:                                  # keep going; the traceback says what to fix
            import traceback
            traceback.print_exc()
            print(f"  FAILED: {k}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/figs11.py
"""
Paper figures, v11 design system (Journal of Neural Engineering; plan: paper/S2I_figure_plan_v11.md).

Design rules
  * IOP sizes: double column 150 mm, single column 85 mm; text 8-12 pt at print size (panel letter 10 pt bold,
    panel title 9 pt, labels and ticks 8 pt); one sans family
  * axes are placed in millimetres on the page (Canvas.ax), so the layout is fixed and fills the figure
  * short panel titles on the letter line, no subtitles (conditions go to the caption)
  * one semantic palette (ours = ink blue, fixed bank = light blue, baselines = one warm grey, one accent)
  * per-patient points as a deterministic beeswarm; mean + 95 % CI; direct labels, no legends
  * after saving, check_layout() reports overlapping texts and the largest empty area of the page

Each figure reads saved result files only and writes <name>.{pdf,svg,png} plus <name>_numbers.json (every number
printed on the figure). Bootstrap CIs use a seed derived from the comparison name, so they do not depend on the
order in which they are computed.

Usage
  python -m s2i.figs11 --runs RUNS_DIR --out FIG_DIR [--only fig2]
"""
import argparse
import json
import os
import zlib

import numpy as np
import matplotlib as mpl

mpl.use("Agg")
import matplotlib.pyplot as plt                                    # noqa: E402
from matplotlib.lines import Line2D                                # noqa: E402

from s2i import figstyle as st                                     # noqa: E402  (beeswarm, fmt, fonts)
from s2i.paperfigs import (find_leadformer, load_dir, load_strict, paired, STRICT)  # noqa: E402

MM = 1 / 25.4
W2, W1 = 150.0, 85.0
FS, FS_T, FS_L = 8, 9, 10                     # labels/ticks, panel titles, panel letters (pt)

OURS, FIXED, BASE, ACC = "#1f4e79", "#8fb3d9", "#8c8c8c", "#c8553d"
FIXED_TXT = "#4f7fb3"                          # text in the fixed-bank colour (readable on white)
INK, INK2, MUTE = "#1d1d1f", "#5a5a5a", "#d9d9d9"
BASE_PT = "#b9b9b9"                           # baseline patient dots (lighter than the baseline mean)


def setup():
    st.setup()                                # registers Liberation Sans on Colab
    mpl.rcParams.update({
        "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
        "font.size": FS, "axes.labelsize": FS, "xtick.labelsize": FS, "ytick.labelsize": FS,
        "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.major.size": 2.5,
        "ytick.major.size": 2.5, "xtick.major.pad": 2.0, "ytick.major.pad": 2.0, "axes.labelpad": 2.5,
        "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#3a3a3a",
        "xtick.color": "#3a3a3a", "ytick.color": "#3a3a3a", "text.color": INK, "axes.labelcolor": INK,
        "pdf.fonttype": 42, "svg.fonttype": "none", "axes.unicode_minus": True, "savefig.dpi": 600,
        "lines.solid_capstyle": "round"})


def boot_ci(v, name, n=10000):
    """Patient-bootstrap 95 % CI of the mean, seeded by the comparison name (order-independent)."""
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    rng = np.random.default_rng(zlib.crc32(name.encode()))
    m = v[rng.integers(0, len(v), (n, len(v)))].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def paired_stats(a, b, name):
    d = np.asarray(a, float) - np.asarray(b, float)
    lo, hi = boot_ci(d, name)
    return {"mean_diff": float(d.mean()), "ci": [lo, hi], "wins": int((d > 0).sum()), "n": int(len(d))}


fmt = st.fmt


def r_of(d, model, frac="0.2"):
    """Explicit-prediction r if the run stored it for this fraction (as in the manuscript), else the
    sufficient-statistics r; None if the model is missing."""
    ex = d.get("explicit", {}).get(model)
    if ex is not None and abs(ex.get("frac", 0.2) - float(frac)) < 1e-9:
        return ex["mean"]["r"]
    m = d.get("fracs", {}).get(frac, {}).get("models", {}).get(model)
    return m["r"] if m else None


def model_rx(D, model, frac="0.2"):
    out = {s: r_of(d, model, frac) for s, d in D.items()}
    return {s: v for s, v in out.items() if v is not None}


def frac_runs(main_run, name="readout_combo5_strict"):
    """{fraction: {subject: result}} with each calibration fraction read from its own run folder."""
    base = os.path.join(main_run, name + STRICT)
    return {"0.2": load_dir(base, "readout_"), "0.1": load_dir(base + "_f10", "readout_"),
            "0.05": load_dir(base + "_f05", "readout_")}


# ----------------------------------------------------------------------------- canvas in millimetres
class Canvas:
    def __init__(self, w, h):
        self.w, self.h = w, h
        self.fig = plt.figure(figsize=(w * MM, h * MM))

    def ax(self, x, y, w, h):
        """Axes with lower-left corner (x, y) and size (w, h) in mm from the lower-left page corner."""
        return self.fig.add_axes([x / self.w, y / self.h, w / self.w, h / self.h])

    def text(self, x, y, s, **kw):
        kw.setdefault("fontsize", FS)
        return self.fig.text(x / self.w, y / self.h, s, **kw)

    def letter(self, x, y, s, title=None):
        self.text(x, y, s, fontsize=FS_L, fontweight="bold", va="baseline")
        if title:
            self.text(x + 4.6, y, title, fontsize=FS_T, va="baseline")

    def key(self, x, y, items, gap=3.0, text_colors=None):
        """Inline key: items = [(label, colour, marker)], laid out left to right from (x, y) in mm."""
        r = self.fig.canvas.get_renderer()
        for lab, col, mk in items:
            self.fig.lines.append(Line2D([(x + 1.0) / self.w], [(y + 1.0) / self.h], marker=mk,
                                         ms=4 if mk in "oDs" else 6, mew=1.3, color=col,
                                         transform=self.fig.transFigure))
            t = self.text(x + 2.6, y, lab, color=(text_colors or {}).get(lab, col), va="baseline")
            x += 2.6 + t.get_window_extent(r).width / self.fig.dpi * 25.4 + gap


def check_layout(fig, name, cell=4.0):
    """Report overlapping text boxes and the largest empty square of the page (mm), from the rendered image."""
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(mpl.text.Text) if t.get_visible() and t.get_text().strip()]
    boxes = [(t, t.get_window_extent(r)) for t in texts]
    gap = 0.3 * fig.dpi / 25.4                                # minimum clearance between texts (0.3 mm)
    over = []
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            a, b = boxes[i][1], boxes[j][1]
            if min(a.x1, b.x1) - max(a.x0, b.x0) > -gap and min(a.y1, b.y1) - max(a.y0, b.y0) > 1:
                over.append((boxes[i][0].get_text()[:30], boxes[j][0].get_text()[:30]))
    img = np.asarray(fig.canvas.buffer_rgba())[..., :3]
    dpi = fig.dpi
    px = max(1, int(round(cell * dpi / 25.4)))
    H, W = img.shape[0] // px, img.shape[1] // px
    ink = np.array([[(img[i * px:(i + 1) * px, j * px:(j + 1) * px] < 245).any() for j in range(W)]
                    for i in range(H)])
    best = np.zeros((H + 1, W + 1), int)                      # largest all-empty square (DP)
    for i in range(H):
        for j in range(W):
            if not ink[i, j]:
                best[i + 1, j + 1] = 1 + min(best[i, j + 1], best[i + 1, j], best[i, j])
    side = best.max() * cell
    print(f"  [{name}] layout: {len(over)} text overlaps; largest empty square {side:.0f} mm; "
          f"empty cells {100 * (1 - ink.mean()):.0f} %")
    for a, b in over:
        print(f"     overlap: '{a}' / '{b}'")
    return {"overlaps": over, "largest_empty_square_mm": side, "empty_fraction": float(1 - ink.mean())}


def save(cv, out, name, nums):
    os.makedirs(out, exist_ok=True)
    nums["_layout"] = check_layout(cv.fig, name)
    for ext in ("pdf", "svg", "png"):
        cv.fig.savefig(os.path.join(out, f"{name}.{ext}"))
    plt.close(cv.fig)
    json.dump(nums, open(os.path.join(out, f"{name}_numbers.json"), "w"), indent=1)
    print("  ->", os.path.join(out, name + ".{pdf,svg,png}"))


def tl(v, d=2):
    """Tick label: shortest decimal up to d places, leading zero, true minus ("0.2", "0.15", "\u22120.1")."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.{d}f}".rstrip("0").rstrip(".")
    return s.replace("-", "\u2212")


def nice_ticks(lo, hi, maxn=4, steps=(0.01, 0.02, 0.04, 0.05, 0.1, 0.2, 0.25, 0.5, 1.0)):
    """Ticks inside [lo, hi] on the finest step that gives at most maxn ticks."""
    for s in steps:
        v = np.arange(np.ceil(lo / s - 1e-9), np.floor(hi / s + 1e-9) + 1) * s
        if len(v) <= maxn:
            return np.round(v, 6)
    return np.round(v, 6)


def cat_axis(ax):
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0, pad=2)


# ----------------------------------------------------------------------------- Fig. 2
def fig2(runs, out):
    main_run, noenv = os.path.join(runs, "shared_bip_car"), os.path.join(runs, "shared_no_env")
    S5 = load_strict(main_run)
    SF = load_dir(os.path.join(noenv, "readout_fixed_strict" + STRICT), "readout_")
    if not S5 or not SF:
        raise FileNotFoundError(2, "missing", os.path.join(main_run, "readout_combo5_strict" + STRICT))
    subs, (base, fixed, final) = paired(model_rx(S5, "base"), model_rx(SF, "combo_nosess"), model_rx(S5, "combo_nosess"))
    gain = paired_stats(final, base, "fig2a final-base primary strict")
    nums = {"a": {"n": len(subs), "mean_base": base.mean(), "mean_fixed": fixed.mean(), "mean_final": final.mean(),
                  "final_minus_base": gain}}

    cv = Canvas(W2, 96)

    # ---------------- a: dumbbell per patient, sorted by gain; gain bars; mean row
    order = np.argsort(final - base)
    yy = np.arange(len(subs)) + 1.4
    ax = cv.ax(10, 10, 44, 75)
    for y, i in zip(yy, order):
        ax.plot([base[i], final[i]], [y, y], color=MUTE, lw=2.2, zorder=1)
        ax.plot(fixed[i], y, marker="|", ms=6, mew=1.3, color=FIXED, zorder=2)
        ax.plot(base[i], y, "o", ms=3.6, color=BASE, zorder=3)
        ax.plot(final[i], y, "o", ms=3.8, color=OURS, zorder=4)
    mb, mx, mf = base.mean(), fixed.mean(), final.mean()
    ax.plot([mb, mf], [0, 0], color=OURS, lw=2.6, alpha=0.25)
    ax.plot(mx, 0, marker="|", ms=7, mew=1.6, color=FIXED)
    ax.plot(mb, 0, "o", ms=4.6, color=BASE)
    ax.plot(mf, 0, "o", ms=4.8, color=OURS)
    ax.axhline(0.7, color=MUTE, lw=0.6)
    lab = [s.replace("sub-", "P") for s in subs]
    ax.set_yticks([0] + list(yy), ["mean"] + [lab[i] for i in order])
    ax.get_yticklabels()[0].set_fontweight("bold")
    lo = np.floor(min(base.min(), fixed.min()) * 20) / 20
    hi = np.ceil(max(final.max(), fixed.max()) * 20) / 20
    ax.set_xticks(np.round(np.arange(np.ceil(lo * 10) / 10, hi + 1e-9, 0.1), 1))
    ax.set_xlim(lo - 0.01, hi + 0.005)
    ax.set_xlabel("test r (mean over contacts)")
    ax.set_ylim(-0.6, yy[-1] + 0.6)
    cat_axis(ax)
    cv.key(10, 87.0, [("LAG9 ridge", BASE, "o"), ("fixed bank", FIXED, "|"), ("final model", OURS, "o")], gap=4,
           text_colors={"fixed bank": FIXED_TXT})

    g = (final - base)[order]
    axg = cv.ax(59, 10, 12, 75)
    axg.barh(yy, g, height=0.62, color=ACC, alpha=0.85)
    lo_ci, hi_ci = gain["ci"]
    axg.errorbar(gain["mean_diff"], 0, xerr=[[gain["mean_diff"] - lo_ci], [hi_ci - gain["mean_diff"]]], fmt="s",
                 ms=4, color=ACC, capsize=0, lw=1.2)
    gmax = max(0.08, np.ceil(g.max() * 50) / 50)
    axg.text(hi_ci + 0.004, 0, fmt(gain["mean_diff"], sign=True), color=ACC, va="center", fontweight="bold")
    axg.set_ylim(ax.get_ylim())
    axg.set_xlim(min(0, g.min() * 1.1), gmax)
    axg.set_xticks([0, gmax], ["0", f"{gmax:.2f}"])
    axg.set_yticks([])
    axg.spines["left"].set_visible(False)
    axg.axvline(0, color=INK2, lw=0.6)
    axg.axhline(0.7, color=MUTE, lw=0.6)
    axg.set_xlabel("final − LAG9 ridge")
    cv.letter(2, 91.5, "a", "Every patient improves")
    nums["a"]["gain_axis_max"] = gmax

    # ---------------- b: all models on the same trials (secondary level, session-wise normalisation)
    T5 = load_dir(os.path.join(main_run, "readout_combo5"), "readout_")
    TF = load_dir(os.path.join(noenv, "readout_fixed"), "readout_")
    nfn = {s: d["models"]["nfn_mode"]["r"] for s, d in load_dir(os.path.join(runs, "nfn_baseline"), "nfn_").items()}
    rows = [("final model", model_rx(T5, "combo"), OURS), ("fixed bank", model_rx(TF, "combo"), FIXED),
            ("LAG9 ridge", model_rx(T5, "base"), BASE), ("LeadFormer", find_leadformer(runs), BASE),
            ("NeuroFlowNet", nfn, BASE)]
    rows = [r for r in rows if r[1]]
    axb = cv.ax(103, 55, 35, 31)
    nums["b"] = {}
    xmax = 0.0
    for k, (name, d, col) in enumerate(rows):
        y = len(rows) - 1 - k
        v = np.array(list(d.values()), float)
        off = st.beeswarm(v, size_data=0.006, max_off=0.30)
        axb.scatter(v, y + off, s=5, color=col if col != BASE else BASE_PT, alpha=0.75 if col == OURS else 0.9,
                    lw=0, zorder=2)
        lo_, hi_ = boot_ci(v, f"fig2b {name}")
        mc = {BASE: INK2, FIXED: FIXED_TXT}.get(col, col)
        axb.plot([lo_, hi_], [y, y], color=mc, lw=1.1, zorder=3)
        axb.plot(v.mean(), y, "D", ms=4.2, color=mc, mec="white", mew=0.6, zorder=4)
        axb.text(1.03, y, fmt(v.mean()), va="center", ha="left", color=mc, transform=axb.get_yaxis_transform(),
                 fontweight="bold" if k == 0 else "normal")
        nums["b"][name] = {"n": len(v), "mean": float(v.mean()), "ci": [lo_, hi_]}
        xmax = max(xmax, np.ceil(v.max() * 20) / 20 + 0.01)
    axb.set_yticks(range(len(rows)), [r[0] for r in rows][::-1])
    for t, r in zip(axb.get_yticklabels(), rows[::-1]):
        t.set_color({BASE: INK2, FIXED: FIXED_TXT}.get(r[2], r[2]))
    axb.set_xlim(0, xmax)
    xv = np.round(np.arange(0, xmax + 1e-9, 0.1), 1)
    axb.set_xticks(xv, [tl(v) for v in xv])
    axb.set_ylim(-0.6, len(rows) - 0.4)
    cat_axis(axb)
    axb.set_xlabel("test r per patient")
    cv.letter(76, 91.5, "b", "Comparison with baselines")

    # ---------------- c: calibration efficiency (primary level, strict)
    FR = frac_runs(main_run)
    fr = [f for f in ("0.05", "0.1", "0.2") if FR[f]]
    subs_c = sorted(set.intersection(*[set(FR[f]) for f in fr]))
    xm = np.array([np.mean([FR[f][s]["fracs"][f]["minutes"] for s in subs_c]) for f in fr])
    Y = {k: np.array([[r_of(FR[f][s], k, f) for f in fr] for s in subs_c], float)
         for k in ("base", "combo_nosess")}
    axc = cv.ax(88, 10, 50, 31)
    nums["c"] = {"minutes": xm.tolist()}
    for k, col, name, dx in (("base", BASE, "ridge", -0.07), ("combo_nosess", OURS, "final", 0.07)):
        mu = Y[k].mean(0)
        ci = np.array([boot_ci(Y[k][:, j], f"fig2c {k} {f}") for j, f in enumerate(fr)])
        for x_, (l_, h_) in zip(xm + dx, ci):                      # thin CI whiskers, models nudged apart
            axc.plot([x_, x_], [l_, h_], color=col, lw=0.8, alpha=0.6, solid_capstyle="butt", zorder=2)
        axc.plot(xm, mu, "-o", color=col, ms=3.8, lw=1.4, zorder=3)
        axc.text(xm[-1] + 0.25, mu[-1], {"ridge": "LAG9\nridge"}.get(name, name), color=col, va="center", linespacing=1.0)
        nums["c"][name] = mu.tolist()
        nums["c"][name + "_ci"] = ci.tolist()
    if "0.1" in fr and "0.2" in fr:
        j10, j20 = fr.index("0.1"), fr.index("0.2")
        s_ = paired_stats(Y["combo_nosess"][:, j10], Y["base"][:, j20], "fig2c final10-base20 primary strict")
        nums["c"]["final10_minus_base20"] = s_
        y10, yb20 = Y["combo_nosess"][:, j10].mean(), Y["base"][:, j20].mean()
        axc.plot([xm[j10], xm[j20]], [yb20, yb20], color=ACC, lw=0.8, ls=(0, (2, 1.5)), zorder=2)
        axc.plot([xm[j10], xm[j10]], [yb20, y10], color=ACC, lw=1.4, zorder=2)
        axc.text(xm[j10] + 0.25, np.min(nums["c"]["ridge_ci"]) + 0.002,
                 f"final {xm[j10]:.1f} min − LAG9 ridge {xm[j20]:.1f} min\n"
                 f"{fmt(s_['mean_diff'], sign=True)} [{fmt(s_['ci'][0])}, {fmt(s_['ci'][1])}], "
                 f"{s_['wins']}/{s_['n']}", color=ACC, va="bottom", ha="left", linespacing=1.25)
    axc.set_xticks(xm, [f"{v:.1f}" for v in xm])
    axc.set_xlim(xm[0] - 0.3, xm[-1] + 1.2)
    allci = np.concatenate([np.ravel(nums["c"][k]) for k in nums["c"] if k.endswith("_ci")])
    ylo, yhi = np.floor(allci.min() * 50) / 50, np.ceil(allci.max() * 50) / 50
    axc.set_ylim(ylo - 0.004, yhi + 0.004)
    yv = np.round(np.arange(np.ceil(ylo * 20) / 20, yhi + 1e-9, 0.05), 2)
    axc.set_yticks(yv, [fmt(v, 2) for v in yv])
    axc.set_xlabel("calibration (min)")
    axc.set_ylabel("test r")
    cv.letter(76, 45.5, "c", "Calibration efficiency")

    save(cv, out, "fig2", nums)


# ----------------------------------------------------------------------------- Fig. 1 (overview)
DATA_DIR = None                               # set by main(); one patient file gives real shafts, traces, sessions
FIG1_PATIENT = "sub-12"
TEN20 = {"Fp1": (-29, 84, -7), "Fp2": (30, 85, -7), "F7": (-70, 42, -11), "F3": (-50, 53, 42), "Fz": (0, 59, 66),
         "F4": (52, 54, 41), "F8": (73, 44, -12), "T7": (-84, -16, -9), "C3": (-65, -12, 64), "Cz": (0, -9, 100),
         "C4": (67, -11, 64), "T8": (85, -15, -9), "P7": (-72, -73, -2), "P3": (-53, -79, 56), "Pz": (0, -81, 83),
         "P4": (56, -79, 57), "P8": (73, -73, -3), "O1": (-29, -112, 9), "O2": (30, -112, 9)}   # MNE standard_1020, mm
SEEG = ACC                                    # Fig. 1 only: intracranial contacts and recorded SEEG
TINT = "#f4f6f9"


def _arrow(ax, p, q, color=INK2, lw=0.75, ls="-", rad=0.0, ms=6, z=5):
    from matplotlib.patches import FancyArrowPatch
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=ms, color=color, lw=lw, linestyle=ls,
                                 connectionstyle=f"arc3,rad={rad}", shrinkA=0, shrinkB=0, zorder=z))


def _rbox(ax, x, y, w, h, fc, ec="none", lw=0.6, r=1.2, z=0):
    from matplotlib.patches import FancyBboxPatch
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}", fc=fc, ec=ec, lw=lw,
                                zorder=z))


def _morlet_bank(n=12, half=240, fs=200.0):
    """Encoder initialisation (s2i.shared.TemporalEncoder): 12 Morlet pairs 2-60 Hz, 5 cycles, +-1.2 s."""
    t = np.arange(-half, half + 1) / fs
    f = np.geomspace(2, 60, n)
    g = np.exp(-t[None] ** 2 / (2 * (5 / (2 * np.pi * f[:, None])) ** 2))
    return t, f, g * np.cos(2 * np.pi * f[:, None] * t[None])


def _fig1_inputs():
    """Real inputs if DATA_DIR / runs are available, otherwise illustrative stand-ins (flagged in the numbers)."""
    rng = np.random.default_rng(7)
    src = {"real": []}
    shafts, eeg, seeg, sessions, kernels = None, None, None, None, None
    p = os.path.join(DATA_DIR or "", f"{FIG1_PATIENT}.pt")
    if DATA_DIR and os.path.exists(p):
        import torch
        d = torch.load(p, weights_only=False, map_location="cpu")
        pos, names = np.asarray(d["contact_mni"], float), list(d["contact_names"])
        sh = {}
        for n_, xyz in zip(names, pos):
            sh.setdefault(n_.rstrip("0123456789"), []).append(xyz)
        shafts = [np.array(v) for v in sh.values() if len(v) >= 3]
        b = np.asarray(d["trial_bounds"])
        s0, s1 = b[len(b) // 2]
        eeg = np.asarray(d["eeg_car"][:, s0:s0 + 400], float)[:4]
        seeg = np.asarray(d["ieeg_bip"][:, s0:s0 + 400], float)[:3]
        ses = list(d.get("trial_session", []))
        sessions = [ses.count(s) for s in dict.fromkeys(ses)]
        src["real"] += ["shafts", "traces", "sessions"]
    if shafts is None:                        # illustrative: orthogonal shafts in the left hemisphere
        shafts = []
        for y, z in ((25, 5), (-5, -15), (-25, -18), (-40, 10), (5, 35), (-60, 20)):
            x = np.linspace(-66, -22, 8)
            shafts.append(np.c_[x, np.full(8, y) + rng.normal(0, 1, 8), np.full(8, z) + np.linspace(0, -3, 8)])
    if eeg is None:
        t = np.arange(400) / 200
        eeg = np.array([np.sin(2 * np.pi * (6 + 2 * k) * t + k) * 0.6 + np.cumsum(rng.normal(0, 0.25, 400)) * 0.08
                        + rng.normal(0, 0.25, 400) for k in range(4)])
        lat = np.array([np.convolve(rng.normal(0, 1, 400), np.ones(15) / 15, "same") for _ in range(3)])
        seeg = lat * 3
    if sessions is None:
        sessions = [62, 55, 60, 48]
    t, f, kernels = _morlet_bank()
    return src, shafts, eeg, seeg, sessions, (t, f, kernels)


def _trace(ax, x0, y0, w, h, v, color, lw=0.7, z=4):
    v = np.asarray(v, float)
    v = (v - v.mean()) / (np.abs(v - v.mean()).max() + 1e-9)
    ax.plot(x0 + np.linspace(0, w, len(v)), y0 + v * h / 2, color=color, lw=lw, zorder=z, solid_joinstyle="round")


def _bandpass(x, f0, fs=200.0, q=2.5):
    """Zero-phase FFT band-pass around f0 (Gaussian, width f0 / q) - only for drawing the fan-out."""
    X = np.fft.rfft(x - x.mean())
    f = np.fft.rfftfreq(len(x), 1 / fs)
    return np.fft.irfft(X * np.exp(-0.5 * ((f - f0) / max(f0 / q, 1.0)) ** 2), len(x))


def _sheet(ax, x0, y0, nx, ny, du, dv, vals, cmap, z, ec="white"):
    """One feature sheet in oblique projection: nx time cells along (du, 0), ny filter cells along dv."""
    from matplotlib.collections import PolyCollection
    polys, cols = [], []
    for i in range(ny):
        for j in range(nx):
            o = np.array([x0 + j * du + i * dv[0], y0 + i * dv[1]])
            polys.append([o, o + [du, 0], o + [du + dv[0], dv[1]], o + [dv[0], dv[1]]])
            cols.append(cmap(vals[i, j]))
    ax.add_collection(PolyCollection(polys, facecolors=cols, edgecolors=ec, linewidths=0.25, zorder=z))


def fig1(runs, out):
    from s2i import render3d as r3
    from matplotlib.colors import LinearSegmentedColormap
    W, H = W2, 62.0
    cv = Canvas(W, H)
    ax = cv.ax(0, 0, W, H)
    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.axis("off")
    src, shafts, eeg, seeg, sessions, (kt, kf, kern) = _fig1_inputs()
    nums = {"inputs": src, "patient": FIG1_PATIENT, "sessions": sessions}
    TY = 58.0
    cv.letter(0.5, TY, "a", "Data and protocol")
    cv.letter(37.0, TY, "b", "Model for a new patient")
    cv.letter(120.5, TY, "c", "Meta-training")
    blues = mpl.colormaps["Blues"]

    # ---------------- a: head outline, scalp electrodes, brain with SEEG shafts; session timeline
    br = r3.load_brain()
    R = r3.camera(140, 42)
    s, cen = 0.165, np.array([17.6, 35.0])

    def to_mm(p2):
        return s * p2 + cen
    head = r3.ellipsoid_outline(R, np.array([0.0, -14.0, 18.0]), np.array([88.0, 108.0, 96.0]), s, cen)
    ax.plot(head[:, 0], head[:, 1], color="#c9ced6", lw=0.7, zorder=1.5)
    parts = []
    for h_ in ("left", "right"):
        v, f, su = br[h_]
        t_ = 1 / (1 + np.exp(-su[f].mean(1) / 0.6))
        parts.append(r3.mesh_faces(v, f, R, to_mm, r3.mix("#ece8e2", "#c9c2b8", t_), vn=r3.vertex_normals(v, f),
                                   ambient=0.42, spec=0.06))
    r3.draw_faces(ax, parts, z=2)
    fwd = R[2]
    for k, p3 in TEN20.items():
        p3 = np.array(p3, float) + np.array([0, -4, -10])
        if (p3 - np.array([0, -15, 20])) @ fwd > -5:
            ax.plot(*to_mm(R[:2] @ p3), "o", ms=3.3, color=INK2, mec="white", mew=0.5, zorder=6)
    for sh in shafts:
        xy = to_mm(sh @ R[:2].T)
        ax.plot(xy[:, 0], xy[:, 1], color="white", lw=2.0, zorder=7)
        ax.plot(xy[:, 0], xy[:, 1], color=INK2, lw=0.6, zorder=8)
        ax.plot(xy[:, 0], xy[:, 1], "o", ms=1.8, color=SEEG, mew=0, zorder=9)
    ax.plot(1.6, 15.6, "o", ms=3.3, color=INK2, mec="white", mew=0.5)
    ax.text(3.0, 15.6, "scalp EEG", color=INK2, va="center")
    ax.plot(21.4, 15.6, "o", ms=2.2, color=SEEG, mew=0)
    ax.text(22.8, 15.6, "SEEG", color=SEEG, va="center")
    x0, x1, yb, hb = 0.8, 34.5, 4.2, 3.0
    tot, ncal = sum(sessions), int(round(0.2 * sum(sessions)))
    scale = (x1 - x0 - 1.0 * (len(sessions) - 1)) / tot
    xx, k = x0, 0
    for ns in sessions:
        for j in range(ns):
            col = OURS if k < ncal else ("white" if k == ncal else "#b9b9b9")
            ax.add_patch(mpl.patches.Rectangle((xx + j * scale, yb), scale * 1.02, hb, color=col, lw=0, zorder=3))
            k += 1
        xx += ns * scale + 1.0
    ax.text(x0, yb + hb + 0.9, "calibration", color=OURS, va="bottom")
    ax.text(x1, yb + hb + 0.9, "test", color=INK2, va="bottom", ha="right")
    ax.text(x0, yb - 0.9, "first 20 % of trials", color=OURS, va="top")

    # ---------------- b: the computation  y_k(t) = sum_c sum_f W_kcf (h_f * x_c)(t)
    top, bot = 51.0, 25.0                     # vertical extent of the main row
    lab_y = bot - 1.6
    mid = (top + bot) / 2
    # (1) multichannel scalp EEG, one channel highlighted
    ex0, ex1 = 38.5, 49.0
    nch = len(eeg)
    eeg_y = np.linspace(top - 2.5, bot + 2.5, nch)
    hi_c = 1
    for i, (v, y_) in enumerate(zip(eeg, eeg_y)):
        _trace(ax, ex0, y_, ex1 - ex0, 4.0, v, OURS if i == hi_c else "#9a9a9a", lw=0.8 if i == hi_c else 0.55)
    ax.text(ex0, lab_y, "scalp EEG\n" + r"$x_c(t)$", va="top", linespacing=1.15)
    # (2) fan-out of the highlighted channel through the shared filter bank
    fx0, fx1 = 54.0, 66.0
    freqs = [2.0, 5.0, 12.0, 25.0, 50.0]
    fy = np.linspace(top - 2.0, bot + 2.0, len(freqs))
    for i, (f0, y_) in enumerate(zip(freqs, fy)):
        col = blues(0.45 + 0.5 * i / (len(freqs) - 1))
        ax.plot([ex1 + 0.6, fx0 - 0.8], [eeg_y[hi_c], y_], color=col, lw=0.6, zorder=2)
        _trace(ax, fx0, y_, fx1 - fx0, 3.4, _bandpass(np.asarray(eeg[hi_c], float), f0), col, lw=0.7)
    ax.text(fx1 + 0.5, fy[0], "2 Hz", va="center", color=INK2)
    ax.text(fx1 + 0.5, fy[-1], "60 Hz", va="center", color=INK2)
    ax.text(fx0, lab_y, "filter bank\n" + r"$h_f * x_c$", va="top", linespacing=1.15)
    # (3) feature tensor: one sheet per scalp channel (filters x time)
    rng = np.random.default_rng(5)
    tx0, ty0 = 76.0, 28.0
    du, dv, off = 0.72, (0.30, 1.45), 1.5
    for c in range(4)[::-1]:
        vals = np.clip(np.abs(rng.normal(0.35, 0.22, (12, 9))) + np.linspace(0, 0.25, 9)[None], 0, 1)
        _sheet(ax, tx0 + off * c, ty0 + off * c, 9, 12, du, dv, vals * (0.55 + 0.45 * (c == hi_c)) + 0.05,
               blues, z=3 + (3 - c))
    ax.text(tx0, lab_y, "features\n" + r"$\phi_{cf}(t)$", va="top", linespacing=1.15)
    # (4) patient-specific readout W
    wx0, wx1, wy0, wy1 = 93.0, 98.5, 29.0, 48.0
    div = LinearSegmentedColormap.from_list("d", [OURS, "#f7f7f7", SEEG])
    ax.imshow(rng.normal(0, 1, (12, 4)), extent=(wx0, wx1, wy0, wy1), cmap=div, aspect="auto", zorder=3,
              vmin=-2.4, vmax=2.4, interpolation="nearest")
    ax.text((wx0 + wx1) / 2, lab_y, "readout\n" + r"$W_{kcf}$", va="top", ha="center", linespacing=1.15)
    # (5) SEEG shaft with contacts and recorded / predicted traces
    sx = 102.0
    cy = np.linspace(top - 1.0, bot + 1.0, 7)
    ax.plot([sx, sx], [cy[-1] - 1.5, cy[0] + 1.5], color=INK2, lw=0.8, zorder=3)
    ax.plot(np.full(len(cy), sx), cy, "o", ms=2.6, color="white", mec=SEEG, mew=0.8, zorder=4)
    for j, i in enumerate([1, 3, 5]):
        ax.plot(sx, cy[i], "o", ms=2.6, color=SEEG, mew=0, zorder=5)
        v = np.asarray(seeg[j], float)
        pred = np.convolve(v, np.ones(9) / 9, "same") * 0.8
        _trace(ax, sx + 1.8, cy[i], 13.6, 4.2, v, SEEG, lw=0.6)
        _trace(ax, sx + 1.8, cy[i], 13.6, 4.2, pred, OURS, lw=0.9, z=5)
    ax.text(sx + 1.8, top + 3.8, "recorded", color=SEEG, va="bottom")
    ax.text(sx + 1.8, top + 0.7, "predicted", color=OURS, va="bottom")
    ax.text(sx + 8.6, lab_y, "bipolar SEEG\n" + r"$y_k(t)$", va="top", ha="center", linespacing=1.15)
    for xa, xb in ((fx1 + 6.4, tx0 - 0.6), (tx0 + 13.4, wx0 - 0.6), (wx1 + 0.6, sx - 1.4)):
        _arrow(ax, (xa, mid), (xb, mid))
    # the model in one line, with what is shared and what is fitted per patient
    ax.text(78.0, 10.4, r"$\hat y_k(t)=\sum_{c}\sum_{f}\,W_{kcf}\,(h_f * x_c)(t)$", ha="center", va="center",
            fontsize=FS_T)
    for (xa, xb), lab, col in (((fx0, fx1 + 6.0), r"$h_f$: shared across patients", OURS),
                               ((wx0 - 1.0, 117.4), r"$W$: ridge, per patient", INK2)):
        yb_ = 5.6
        ax.plot([xa, xa, xb, xb], [yb_ + 1.1, yb_, yb_, yb_ + 1.1], color=col, lw=0.7)
        ax.text((xa + xb) / 2, yb_ - 0.9, lab, color=col, ha="center", va="top")

    # ---------------- c: episodic meta-training through a closed-form ridge readout
    cx0, cx1 = 121.5, 146.8
    cm = (cx0 + cx1) / 2
    for i, j in enumerate([0, 3, 6, 9]):
        kk = kern[j][np.abs(kern[j]) > 0.02 * np.abs(kern[j]).max()]
        _trace(ax, cx0 + 0.3 + i * 6.3, 51.0, 5.4, 2.8, kk, blues(0.5 + 0.15 * i), lw=0.7)
    ax.text(cx0, 46.6, r"shared filters $h_f$", color=OURS, va="center")
    _arrow(ax, (cm, 44.4), (cm, 41.8))
    for i in range(3)[::-1]:
        _rbox(ax, cx0 + 1.2 * i, 32.2 - 1.2 * i, cx1 - cx0 - 2.4, 8.4, "white", ec="#c9ced6", lw=0.5, r=0.8, z=3 + i)
    xl, wcard = cx0 + 2.4, cx1 - cx0 - 4.8
    ax.text(xl, 36.4, "training patient", va="center", zorder=8)
    ax.add_patch(mpl.patches.Rectangle((xl, 31.6), wcard * 0.64, 1.8, color=OURS, lw=0, zorder=8))
    ax.add_patch(mpl.patches.Rectangle((xl + wcard * 0.70, 31.6), wcard * 0.30, 1.8, color=SEEG, lw=0, zorder=8))
    ax.text(xl, 31.0, "support", color=OURS, va="top", zorder=8)
    ax.text(xl + wcard, 31.0, "query", color=SEEG, va="top", ha="right", zorder=8)
    _arrow(ax, (cm, 26.6), (cm, 24.2))
    _rbox(ax, cx0, 15.4, cx1 - cx0, 8.6, "white", ec="#c9ced6", lw=0.5, r=0.8, z=3)
    ax.text(cm, 21.6, "closed-form ridge", ha="center", va="center", zorder=4)
    ax.text(cm, 17.8, r"$W=(X^{T}X+\lambda I)^{-1}X^{T}Y$", ha="center", va="center", zorder=4)
    _arrow(ax, (cm, 15.4), (cm, 13.0))
    lt = ax.text(cm - 2.0, 10.4, r"loss $=1-r_{\mathrm{query}}$", ha="center", va="center", color=SEEG)
    cv.fig.canvas.draw()
    bb = lt.get_window_extent(cv.fig.canvas.get_renderer()).transformed(ax.transData.inverted())
    xg = cx1 + 2.2
    _arrow(ax, (xg, 10.4), (xg, 50.6), color=ACC, lw=0.9, ls=(0, (3, 2)))
    ax.plot([bb.x1 + 0.8, xg], [10.4, 10.4], color=ACC, lw=0.9, ls=(0, (3, 2)))
    ax.text(xg, 5.4, "gradient updates " + r"$h_f$", color=ACC, ha="right", va="center")

    save(cv, out, "fig1", nums)


# ----------------------------------------------------------------------------- Fig. 3 (what transfers)
def _init_pairs(cfg, fs=200.0):
    """Initial quadrature pairs of the temporal encoder (same formula as s2i.shared.TemporalEncoder)."""
    half, n = cfg.get("half", 240), cfg.get("n_pairs", 12)
    t = np.arange(-half, half + 1) / fs
    f = np.geomspace(cfg.get("fmin", 2.0), cfg.get("fmax", 60.0), n)
    g = np.exp(-t[None] ** 2 / (2 * (cfg.get("cycles", 5.0) / (2 * np.pi * f[:, None])) ** 2))
    w = np.zeros((2 * n, t.size))
    w[0::2] = g * np.cos(2 * np.pi * f[:, None] * t[None])
    w[1::2] = g * np.sin(2 * np.pi * f[:, None] * t[None])
    return w


def _pair_response(W, n, nfft=4096, fs=200.0):
    """Quadrature-pair amplitude response sqrt(|H_cos|^2 + |H_sin|^2), peak-normalised per pair."""
    f = np.fft.rfftfreq(nfft, 1 / fs)
    H = np.sqrt(np.abs(np.fft.rfft(W[0:2 * n:2], nfft)) ** 2 + np.abs(np.fft.rfft(W[1:2 * n:2], nfft)) ** 2)
    return f, H / H.max(1, keepdims=True)


def _bank(main_run):
    """(f, H0 initial, Hs learned per fold) from the encoder checkpoints enc_sub-*.pt (one per held-out patient)."""
    import glob
    encs = sorted(glob.glob(os.path.join(main_run, "enc_sub-*.pt")))
    if not encs:
        raise FileNotFoundError(2, "missing", os.path.join(main_run, "enc_sub-*.pt"))
    import torch
    cks = [torch.load(p, map_location="cpu", weights_only=False) for p in encs]
    cfg = cks[0]["cfg"]
    n = cfg.get("n_pairs", 12)
    f, H0 = _pair_response(_init_pairs(cfg), n)
    Hs = np.array([_pair_response(c["state"]["conv.weight"].numpy()[:, 0], n)[1] for c in cks])
    return f, H0, Hs, [os.path.basename(p)[4:-3] for p in encs]


def _bw(f, H, pk0):
    """Width (octaves) of the contiguous band around each pair's initial peak where the gain is >= -3 dB."""
    out = []
    for k in range(H.shape[0]):
        i = int(np.argmin(np.abs(f - pk0[k])))
        ok = H[k] >= 10 ** (-3 / 20)
        lo, hi = i, i
        while lo > 1 and ok[lo - 1]:
            lo -= 1
        while hi < len(f) - 1 and ok[hi + 1]:
            hi += 1
        out.append(float(np.log2(f[hi] / f[max(lo, 1)])) if ok[i] else float("nan"))
    return out


def _offpeak(f, H, pk0, lo=0.75, hi=1.25):
    """Mean gain (dB re the pair's maximum) 0.75-1.25 octave away from the initial peak, both sides."""
    out = []
    for k in range(H.shape[0]):
        d = np.abs(np.log2(np.maximum(f, 1e-9) / pk0[k]))
        m = (d >= lo) & (d <= hi) & (f >= 0.5) & (f <= 95)
        out.append(float(np.mean(20 * np.log10(H[k, m] + 1e-12))) if m.any() else float("nan"))
    return out


def fig3(runs, out):
    main_run, noenv = os.path.join(runs, "shared_bip_car"), os.path.join(runs, "shared_no_env")
    S5 = load_strict(main_run)
    L = lambda name: load_dir(os.path.join(noenv, name + STRICT), "readout_")   # noqa: E731
    SF, F3, L3 = L("readout_fixed_strict"), L("readout_fixed_arch3_strict"), L("readout_learned_arch3_strict")
    if not (S5 and SF and F3 and L3):
        raise FileNotFoundError(2, "missing", os.path.join(noenv, "readout_*_arch3_strict" + STRICT))
    steps = [("LAG9 ridge", model_rx(S5, "base"), BASE, INK2),
             ("fixed bank", model_rx(SF, "combo_nosess"), FIXED, FIXED_TXT),
             ("3 fixed banks", model_rx(F3, "combo_nosess"), FIXED, FIXED_TXT),
             ("3 learned banks", model_rx(L3, "combo_nosess"), ACC, ACC),
             ("final model", model_rx(S5, "combo_nosess"), OURS, OURS)]
    import glob
    dense = {}                       # Phase 24: ridge on every lag within +-1.2 s, mixed with LAG9 (Table S12)
    for fp in glob.glob(os.path.join(runs, "dense24", "dense24_sub-*.json")):
        d = json.load(open(fp))
        m_ = d["models"]["dense1200+lag9"]
        dense[d["subject"]] = m_.get("r_explicit", m_["r"])      # same scoring as the registered readouts
    if dense:
        steps.insert(1, ("dense-lag ridge", dense, BASE, INK2))
    subs, V = paired(*[s[1] for s in steps])
    m = np.array([v.mean() for v in V])
    tot = paired_stats(V[-1], V[0], "fig3a final-base primary strict")
    nums = {"a": {"n": len(subs), "means": {s[0]: float(v) for s, v in zip(steps, m)}, "steps": {},
                  "final_minus_base": tot}}

    cv = Canvas(W2, 70)
    TOP, BOT = 58.5, 15.0

    # ---------------- a: horizontal waterfall; bar = mean step, dots = previous mean + each patient's step
    ax = cv.ax(19, BOT, 31, TOP - BOT)
    ys = np.arange(len(steps))[::-1].astype(float)               # first step on top
    ax.plot([m[0]] * 2, [ys[0] - 0.22, ys[0] + 0.22], color=BASE, lw=2.4, solid_capstyle="butt", zorder=3)
    ax.text(1.03, ys[0], fmt(m[0]), transform=ax.get_yaxis_transform(), va="center", color=INK2)
    for k in range(1, len(steps)):
        lab, _, col, tcol = steps[k]
        d = V[k] - V[k - 1]
        y, x0, x1 = ys[k], m[k - 1], m[k]
        ax.add_patch(plt.Rectangle((min(x0, x1), y - 0.2), abs(x1 - x0), 0.4, fc=col, ec="none", zorder=3))
        ax.plot([x0, x0], [ys[k - 1] - 0.2, y + 0.2], color=INK2, lw=0.5, ls=(0, (1.5, 1.5)), zorder=1)
        off = st.beeswarm(x0 + d, size_data=0.006, max_off=0.36)
        ax.scatter(x0 + d, y + off, s=4, color=col if col != FIXED else FIXED_TXT, alpha=0.45, lw=0, zorder=2)
        s_ = paired_stats(V[k], V[k - 1], f"fig3a {lab} step primary strict")
        nums["a"]["steps"][lab] = s_
        ax.plot([x0 + s_["ci"][0], x0 + s_["ci"][1]], [y - 0.32] * 2, color=tcol, lw=0.9, solid_capstyle="butt",
                zorder=4)
        hot = col == ACC
        ax.text(1.03, y, f"{fmt(s_['mean_diff'], sign=True)}", transform=ax.get_yaxis_transform(), va="bottom",
                color=tcol, fontweight="bold" if hot else "normal")
        ax.text(1.03, y - 0.06, f"{s_['wins']}/{s_['n']}", transform=ax.get_yaxis_transform(), va="top",
                color=INK2)
    TICK = {"LAG9 ridge": "LAG9 ridge", "dense-lag ridge": "dense-lag\nridge", "fixed bank": "fixed bank",
            "3 fixed banks": "3 fixed\nbanks", "3 learned banks": "3 learned\nbanks", "final model": "final model"}
    ax.set_yticks(ys, [TICK[s[0]] for s in steps], linespacing=1.0)
    for t, s in zip(ax.get_yticklabels(), steps):
        t.set_color(s[3])
    cat_axis(ax)
    pts = np.concatenate([m[:1]] + [V[k] - V[k - 1] + m[k - 1] for k in range(1, len(steps))])
    pad = 0.04 * (pts.max() - pts.min())
    ax.set_xlim(pts.min() - pad, pts.max() + pad)
    xv = nice_ticks(*ax.get_xlim(), maxn=4)
    ax.set_xticks(xv, [fmt(v, 2) for v in xv])
    ax.set_ylim(-0.6, len(steps) - 0.45)
    ax.set_xlabel("test r")
    cv.text(2, 3.0, f"final − LAG9 ridge {fmt(tot['mean_diff'], sign=True)} [{fmt(tot['ci'][0])}, "
            f"{fmt(tot['ci'][1])}], {tot['wins']}/{tot['n']}", color=OURS, fontweight="bold")
    cv.letter(1, 64.5, "a", "Gain decomposition")

    # ---------------- b: learned response of each pair on an axis relative to its initial peak (octaves)
    f, H0, Hs, folds = _bank(main_run)
    H1 = Hs.mean(0)
    n = H0.shape[0]
    sel = (f >= 1.5) & (f <= 80)
    pk0 = f[np.argmax(H0 * sel, 1)]
    og = np.linspace(-1.5, 1.5, 301)
    FL = -30.0                                                   # dB floor of the display

    def rel(H):
        Z = np.full((n, og.size), np.nan)
        for k in range(n):
            fq = pk0[k] * 2 ** og
            ok = (fq >= f[1]) & (fq <= 95)
            Z[k, ok] = np.clip(20 * np.log10(np.interp(fq[ok], f, H[k]) + 1e-12), FL, 0)
        return Z
    G0, G1 = rel(H0), rel(H1)
    D = G1 - G0
    dm = float(max(6, min(24, np.ceil(np.nanpercentile(np.abs(D), 99) / 6) * 6)))
    from matplotlib.colors import LinearSegmentedColormap
    seq = LinearSegmentedColormap.from_list("s2i_seq", ["#ffffff", "#c6d8ec", "#5d8fc4", OURS, "#0d2440"])
    div = LinearSegmentedColormap.from_list("s2i_div", [ACC, "#efc2b6", "#ffffff", "#b4cbe6", OURS])
    for c_ in (seq, div):
        c_.set_bad("#ececec")
    edges = np.array([[og[np.nonzero(r >= -3)[0][0]], og[np.nonzero(r >= -3)[0][-1]]] for r in G0])
    ext = [og[0], og[-1], -0.5, n - 0.5]
    hx, hw, hg = 73.0, 22.0, 2.5
    for j, (Z, cmap, vmin, vmax, lab) in enumerate(((G1, seq, FL, 0, "learned"),
                                                    (D, div, -dm, dm, "learned \u2212 initial"))):
        x_ = hx + j * (hw + hg)
        axh = cv.ax(x_, BOT, hw, TOP - BOT)
        axh.imshow(Z, aspect="auto", origin="lower", extent=ext, cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation="nearest")
        for k in range(n):                                       # initial half-power band of each pair
            for e in edges[k]:
                axh.plot([e, e], [k - 0.5, k + 0.5], color=INK2 if j else "white", lw=0.6, ls=(0, (1.6, 1.2)))
        axh.set_xticks([-1, 0, 1], ["\u22121", "0", "1"])
        axh.set_xlim(ext[0], ext[1])
        if j == 0:
            axh.set_yticks(range(n), [f"{v:.0f}" if v >= 9.5 else f"{v:.1f}" for v in pk0])
            axh.set_ylabel("pair (initial peak, Hz)")
        else:
            axh.set_yticks([])
        for s in ("left", "bottom"):
            axh.spines[s].set_visible(False)
        axh.tick_params(axis="y", length=0)
        cv.text(x_ + hw / 2, TOP + 1.2, lab, ha="center", va="bottom", color=ACC if j else INK)
    cv.text(hx + hw + hg / 2, 7.6, "octaves from initial peak", ha="center", va="baseline")
    for (x0_, w_, cmap, l0, l1) in ((hx + 6, hw - 14, seq, fmt(FL, 0), "0 dB"),
                                    (hx + hw + hg + 6, hw - 15, div, fmt(-dm, 0), fmt(dm, 0, sign=True) + " dB")):
        axk = cv.ax(x0_, 2.4, w_, 1.6)
        axk.imshow(np.linspace(0, 1, 128)[None], aspect="auto", cmap=cmap, extent=[0, 1, 0, 1])
        axk.set_axis_off()
        cv.text(x0_ - 0.8, 2.6, l0, ha="right", va="bottom")
        cv.text(x0_ + w_ + 0.8, 2.6, l1, ha="left", va="bottom")
    cv.letter(62, 64.5, "b", "Shared filter bank")
    nums["b"] = {"n_pairs": int(n), "n_folds": int(len(Hs)), "folds": folds, "diff_scale_db": dm,
                 "initial_half_power_oct": edges.tolist(), "max_abs_diff_db": float(np.nanmax(np.abs(D))),
                 "bw3db_oct_initial": _bw(f, H0, pk0), "bw3db_oct_learned_mean": _bw(f, H1, pk0),
                 "bw3db_oct_learned_median_fold": np.median([_bw(f, H, pk0) for H in Hs], 0).tolist(),
                 "offpeak_db_initial": _offpeak(f, H0, pk0),
                 "offpeak_db_learned_median_fold": np.median([_offpeak(f, H, pk0) for H in Hs], 0).tolist(),
                 "offpeak_db_definition": "mean gain (dB re peak) between 0.75 and 1.25 octave on both sides of "
                                          "the initial peak, frequencies 0.5-95 Hz"}

    # ---------------- c: peak shift per pair (octaves), dots = folds, rows aligned with b
    pk = np.array([f[np.argmax(H * sel, 1)] for H in Hs])
    sh = np.log2(pk / pk0[None])
    med = np.median(sh, 0)
    mx = float(min(1.0, max(0.25, np.ceil(np.percentile(np.abs(sh), 95) * 4) / 4)))
    axc = cv.ax(126.5, BOT, 14.0, TOP - BOT)
    axc.axvline(0, color=INK2, lw=0.6, zorder=1)
    for k in range(n):
        o = st.beeswarm(sh[:, k], size_data=0.02, max_off=0.3)
        ok_ = np.abs(sh[:, k]) <= mx
        axc.scatter(sh[ok_, k], k + o[ok_], s=3, color=OURS, alpha=0.35, lw=0, zorder=2)
        axc.plot([0, med[k]], [k, k], color=OURS, lw=1.0, zorder=3)
        axc.plot(med[k], k, "D", ms=3.2, color=OURS, mec="white", mew=0.5, zorder=4)
        axc.text(1.14, k, f"{pk0[k] * 2 ** med[k]:.0f}" if pk0[k] * 2 ** med[k] >= 9.5 else
                 f"{pk0[k] * 2 ** med[k]:.1f}", transform=axc.get_yaxis_transform(), va="center", color=OURS)
    axc.set_xlim(-mx, mx)
    out_ = np.abs(sh) > mx
    if out_.any():
        kk = np.nonzero(out_)
        axc.scatter(np.clip(sh[kk], -mx, mx) * 0.96, kk[1], s=6, facecolors="none", edgecolors=OURS, lw=0.5, zorder=2,
                    clip_on=False)
    axc.set_xticks([-mx, 0, mx], [fmt(-mx, 2 if mx % 0.5 else 1), "0", fmt(mx, 2 if mx % 0.5 else 1, sign=True)])
    axc.set_ylim(-0.5, n - 0.5)
    axc.set_yticks([])
    axc.spines["left"].set_visible(False)
    axc.set_xlabel("octaves")
    cv.text(141.5, TOP + 1.2, "Hz", ha="left", va="bottom", color=OURS)
    cv.letter(124.5, 64.5, "c", "Peak shift")
    nums["c"] = {"peak_initial_hz": pk0.tolist(), "peak_shift_oct_median": med.tolist(),
                 "peak_learned_hz_median": (pk0 * 2 ** med).tolist()}

    save(cv, out, "fig3", nums)


# ----------------------------------------------------------------------------- Fig. 4 (protocol sensitivity)
# one factor at a time: (label, variant A, metric A, variant B, metric B), as in s2i.protocol.ONE_FACTOR
FIG4_FACTORS = [
    ("random split, 20 %", "rand_trials20", "r_concat", "clean", "r_concat"),
    ("random split, 90 %", "rand_trials90", "r_concat", "chrono90", "r_concat"),
    ("90 % training", "chrono90", "r_concat", "clean", "r_concat"),
    ("1-s segments", "rand_seg80", "r_concat", "rand_trials80", "r_concat"),
    ("monopolar target", "mono_car", "r_concat", "clean", "r_concat"),
    ("original scalp ref.", "bip_orig", "r_concat", "clean", "r_concat"),
    ("target + reference", "mono_orig", "r_concat", "clean", "r_concat"),
    ("flattened metric", "clean", "r_nfn", "clean", "r_concat"),
]
FIG4_KEY = "target + reference"
# composite emulations named by their ingredients: (variant, metric, select patients, ingredients)
FIG4_INGR = ["random 90 %", "flattened r", "monopolar", "original ref.", "1-s segments", "selected pts",
             "noise prediction"]
FIG4_EMUL = [("ours", "clean", "r_concat", False, ()),
             ("r90", "rand_trials90", "r_concat", False, ("random 90 %",)),
             ("r90f", "rand_trials90", "r_nfn", False, ("random 90 %", "flattened r")),
             ("mo", "mono_orig", "r_concat", False, ("monopolar", "original ref.")),
             ("mos", "mono_orig_seg80", "r_concat", True, ("monopolar", "original ref.", "1-s segments",
                                                            "selected pts")),
             ("noise", "rand_trials90", "r_nfn_null", False, ("random 90 %", "flattened r", "noise prediction"))]


def fig4(runs, out):
    per = load_dir(os.path.join(runs, "protocol"), "protocol_")
    if not per:
        raise FileNotFoundError(2, "missing", os.path.join(runs, "protocol", "protocol_sub-*.json"))
    subs = sorted(per)
    R = lambda v, met, m="enc": np.array([per[s][v][m][met] for s in subs], float)   # noqa: E731
    nums = {"n": len(subs), "a": {}, "b": {}, "c": {}}
    cv = Canvas(W2, 66)
    TOP, BOT = 54.5, 10.0

    # ---------------- a: forest plot, one factor at a time (encoder with CI; ridge as open circle)
    ax = cv.ax(25, BOT, 20, TOP - BOT)
    ys = np.arange(len(FIG4_FACTORS))[::-1].astype(float)
    eff = {}
    for y, (lab, va, ma, vb, mb) in zip(ys, FIG4_FACTORS):
        s_ = paired_stats(R(va, ma), R(vb, mb), f"fig4a {lab} enc")
        b_ = float((R(va, ma, "base") - R(vb, mb, "base")).mean())
        eff[lab] = s_["mean_diff"]
        key = lab == FIG4_KEY
        col = ACC if key else OURS
        sig = s_["ci"][0] > 0 or s_["ci"][1] < 0
        ax.plot(s_["ci"], [y, y], color=col, lw=1.6 if key else 1.0, solid_capstyle="butt", zorder=2)
        ax.plot(b_, y, "o", ms=3.4, color=BASE_PT, zorder=3)
        ax.plot(s_["mean_diff"], y, "o", ms=4.6 if key else 3.8, mfc=col if sig else "white", mec=col, mew=1.0,
                zorder=4)
        cv.text(45.6, BOT + (y + 0.6) / (len(FIG4_FACTORS) + 0.2) * (TOP - BOT),
                fmt(s_["mean_diff"], sign=True).replace("−0.000", "0.000"), va="center",
                color=col if key else INK, fontweight="bold" if key else "normal")
        cv.text(63.2, BOT + (y + 0.6) / (len(FIG4_FACTORS) + 0.2) * (TOP - BOT), f"{s_['wins']}/{s_['n']}",
                va="center", ha="right", color=INK2)
        nums["a"][lab] = {**s_, "base_mean_diff": b_}
    add = eff["monopolar target"] + eff["original scalp ref."]
    yk = ys[[f[0] for f in FIG4_FACTORS].index(FIG4_KEY)]
    ax.plot(add, yk, "D", ms=4.0, mfc="white", mec=INK, mew=0.9, zorder=5)
    ax.text(add, yk + 0.32, "sum", ha="center", va="bottom", color=INK)
    nums["a"]["sum_of_single_effects"] = add
    ax.axvline(0, color=INK2, lw=0.6, zorder=0)
    ax.set_yticks(ys, [f[0] for f in FIG4_FACTORS])
    for t, f_ in zip(ax.get_yticklabels(), FIG4_FACTORS):
        if f_[0] == FIG4_KEY:
            t.set_color(ACC)
            t.set_fontweight("bold")
    cat_axis(ax)
    hi = max(v["ci"][1] for k, v in nums["a"].items() if isinstance(v, dict))
    lo = min(v["ci"][0] for k, v in nums["a"].items() if isinstance(v, dict))
    ax.set_xlim(min(-0.03, lo - 0.01), np.ceil(hi * 10) / 10 + 0.01)
    ax.set_xticks(np.round(np.arange(0, np.ceil(hi * 10) / 10 + 1e-9, 0.1), 1),
                  ["0"] + [fmt(v, 1) for v in np.arange(0.1, np.ceil(hi * 10) / 10 + 1e-9, 0.1)])
    ax.set_ylim(-0.6, len(ys) - 0.4)
    ax.set_xlabel("change in test r")
    cv.key(2, 57.0, [("encoder", OURS, "o"), ("LAG9 ridge", BASE_PT, "o")], gap=3, text_colors={"LAG9 ridge": INK2})
    cv.letter(1, 61.5, "a", "One factor at a time")

    # ---------------- b: composite emulations, UpSet style (bars = r, matrix = ingredients)
    rows = []
    for k, v, met, sel, ing in FIG4_EMUL:
        x = R(v, met)
        if sel:
            x = x[np.array([per[s][v]["enc"]["cv_r"] > 0.2 for s in subs])]
        rows.append((k, float(x.mean()), len(x), ing))
    r0 = rows[0][1]
    bx, cw = 91.0, 5.8                                   # first column (mm), column width
    nc = len(rows)
    axb = cv.ax(bx, 35.0, nc * cw, TOP - 35.0)
    for j, (k, v, nn, ing) in enumerate(rows):
        col = OURS if k == "ours" else (MUTE if k == "noise" else (ACC if "monopolar" in ing else BASE))
        axb.bar(j, v, width=0.62, color=col, zorder=2)
        axb.text(j, v + 0.012, fmt(v, 2), ha="center", va="bottom", rotation=90,
                 color={MUTE: INK2, BASE: INK2}.get(col, col))
        nums["b"][k] = {"r": v, "n": nn, "ingredients": list(ing), "x_ours": v / r0}
    axb.set_xlim(-0.5, nc - 0.5)
    ymax = np.ceil(max(r[1] for r in rows) * 10) / 10 + 0.2
    axb.set_ylim(0, ymax)
    yv = np.round(np.arange(0, max(r[1] for r in rows) + 0.05, 0.2), 1)
    axb.set_yticks(yv, [fmt(v, 1) if v else "0" for v in yv])
    axb.set_xticks([])
    axb.spines["bottom"].set_visible(False)
    axb.axhline(0, color=INK2, lw=0.6)
    cv.text(bx - 1.0, TOP + 1.2, "test r", ha="right", va="baseline")
    yx = 32.0                                           # "x ours" row
    cv.text(bx - 1.5, yx, "× ours", ha="right", va="center", color=INK2)
    for j, (k, v, nn, ing) in enumerate(rows):
        s = "1" if k == "ours" else ("–" if k == "noise" else f"{v / r0:.1f}")
        cv.text(bx + (j + 0.5) * cw, yx, s, ha="center", va="center",
                color=ACC if "monopolar" in ing else (OURS if k == "ours" else INK))
    axm = cv.ax(bx, BOT, nc * cw, 19.5)
    ni = len(FIG4_INGR)
    for j, (k, v, nn, ing) in enumerate(rows):
        on = [i for i, g in enumerate(FIG4_INGR) if g in ing]
        axm.scatter([j] * ni, range(ni), s=9, color=MUTE, lw=0, zorder=1)
        if on:
            c_ = ACC if "monopolar" in ing else INK2
            axm.plot([j, j], [min(on), max(on)], color=c_, lw=1.0, zorder=2)
            axm.scatter([j] * len(on), on, s=11, color=c_, lw=0, zorder=3)
    axm.set_xlim(-0.5, nc - 0.5)
    axm.set_ylim(ni - 0.5, -0.5)
    axm.set_yticks(range(ni), FIG4_INGR)
    axm.set_xticks([])
    for s in ("left", "bottom"):
        axm.spines[s].set_visible(False)
    axm.tick_params(axis="y", length=0)
    cv.text(bx - 9.2, TOP - 1.0, "emulations,\nnot\nreproductions", ha="right", va="top",
            color=INK2, linespacing=1.15)
    cv.letter(67, 61.5, "b", "Composite emulations")

    # ---------------- c: per patient, our protocol -> monopolar target + original scalp reference
    a_, b_ = R("clean", "r_concat"), R("mono_orig", "r_concat")
    jump = b_ - a_
    order = np.argsort(jump)
    top = set(order[-4:].tolist())
    axc = cv.ax(132.0, BOT, 14.5, TOP - BOT)
    yy = np.arange(len(subs)).astype(float)
    for y, i in zip(yy, order):
        hot = i in top
        axc.plot([a_[i], b_[i]], [y, y], color=ACC if hot else MUTE, lw=1.4 if hot else 1.2, alpha=0.5 if hot else 1,
                 zorder=1)
        axc.plot(a_[i], y, "o", ms=3.2, color=OURS, zorder=3)
        axc.plot(b_[i], y, "o", ms=3.4, color=ACC if hot else BASE, zorder=3)
    axc.set_yticks(yy, [subs[i].replace("sub-", "P") for i in order])
    for t, i in zip(axc.get_yticklabels(), order):
        if i in top:
            t.set_color(ACC)
            t.set_fontweight("bold")
    cat_axis(axc)
    xm = np.ceil(max(a_.max(), b_.max()) * 5) / 5
    axc.set_xlim(0, xm + 0.02)
    axc.set_xticks(np.round(np.arange(0, xm + 1e-9, 0.4), 1), ["0"] + [fmt(v, 1)
                                                                     for v in np.arange(0.4, xm + 1e-9, 0.4)])
    axc.set_ylim(-0.6, len(subs) - 0.4)
    axc.set_xlabel("test r")
    cv.key(120.5, 57.0, [("ours", OURS, "o"), ("mono + orig.", ACC, "o")], gap=2)
    cv.letter(119.5, 61.5, "c", "Per patient")
    nums["c"] = {"top4": sorted(subs[i] for i in top), "ours": dict(zip(subs, a_.tolist())),
                 "mono_orig": dict(zip(subs, b_.tolist())), "top4_mono_orig_range": [float(b_[list(top)].min()),
                                                                                    float(b_[list(top)].max())]}
    save(cv, out, "fig4", nums)


# ----------------------------------------------------------------------------- Fig. 5 (external validation)
DEV, EXT = INK2, OURS                          # cohort colours in Fig. 5: development (ds004752), external


def _interaction(p_):
    return p_["mono|orig"] - p_["mono|car"] - p_["bip|orig"] + p_["bip|car"]


def _p17_ex(D, m):
    return {s: r["explicit"][m]["mean"]["r"] for s, r in D.items() if m in r.get("explicit", {})}


def fig5(runs, out):
    R = os.path.join(runs, "p17")
    rp = os.path.join(R, "p17_report.json")
    if not os.path.exists(rp):
        raise FileNotFoundError(2, "missing", rp)
    rep = json.load(open(rp))
    desc = {d["comparison"]: d for d in rep.get("descriptive", []) if d.get("level") == "primary" and
            d.get("metric", "r") == "r"}
    cm17 = rep.get("common_mode", {})
    it17 = {s: _interaction(v["pred"]) for s, v in cm17.items()}
    ext = {"h1": rep["H1"], "h2": rep["H2"], "fixed1": desc.get("Final − Fixed-1"),
           "half": next((v for k, v in desc.items() if k.startswith("Final, 10 %")), None)}
    if it17:
        v = np.array(list(it17.values()))
        ext["inter"] = paired_stats(v, np.zeros_like(v), "fig5b interaction ScaleOLDNEW")

    main_run, noenv = os.path.join(runs, "shared_bip_car"), os.path.join(runs, "shared_no_env")
    S5 = load_strict(main_run)
    L = lambda name: load_dir(os.path.join(noenv, name + STRICT), "readout_")   # noqa: E731
    SF, F3, L3 = L("readout_fixed_strict"), L("readout_fixed_arch3_strict"), L("readout_learned_arch3_strict")
    dev = {}
    for k, a_, b_ in (("h1", model_rx(S5, "combo_nosess"), model_rx(S5, "base")),
                      ("h2", model_rx(L3, "combo_nosess"), model_rx(F3, "combo_nosess")),
                      ("fixed1", model_rx(S5, "combo_nosess"), model_rx(SF, "combo_nosess")),
                      ("half", model_rx(frac_runs(main_run)["0.1"], "combo_nosess", "0.1"), model_rx(S5, "base", "0.2"))):
        if a_ and b_:
            _, (x, y) = paired(a_, b_)
            dev[k] = paired_stats(x, y, f"fig5b {k} ds004752 primary strict")
    V = load_dir(os.path.join(runs, "review13"), "volts_")
    cmds = {s: (abs(v["r_scalp_cm_vs_ieeg_cm"]), _interaction(v["pred"])) for s, v in V.items()}
    if cmds:
        v = np.array([c[1] for c in cmds.values()])
        dev["inter"] = paired_stats(v, np.zeros_like(v), "fig5b interaction ds004752")
    nums = {"ds004752": dev, "ScaleOLDNEW": {k: {q: w for q, w in v.items() if q != "per_patient"}
                                             for k, v in ext.items() if v}}
    cv = Canvas(W2, 68)
    TOP, BOT = 56.5, 10.0

    # ---------------- a: H1 per patient (as Fig. 2a), sorted by gain; gain bars
    F = load_dir(os.path.join(R, "main", "final_primary"), "readout_")
    subs, (base, final) = paired(_p17_ex(F, "base"), _p17_ex(F, "combo_nosess"))
    order = np.argsort(final - base)
    yy = np.arange(len(subs)) + 1.4
    ax = cv.ax(9, BOT, 30, TOP - BOT)
    for y, i in zip(yy, order):
        ax.plot([base[i], final[i]], [y, y], color=MUTE, lw=2.2, zorder=1)
        ax.plot(base[i], y, "o", ms=3.6, color=BASE, zorder=3)
        ax.plot(final[i], y, "o", ms=3.8, color=OURS, zorder=4)
    ax.plot([base.mean(), final.mean()], [0, 0], color=OURS, lw=2.6, alpha=0.25)
    ax.plot(base.mean(), 0, "o", ms=4.6, color=BASE)
    ax.plot(final.mean(), 0, "o", ms=4.8, color=OURS)
    ax.axhline(0.7, color=MUTE, lw=0.6)
    ax.set_yticks([0] + list(yy), ["mean"] + [subs[i].replace("sub-", "") for i in order])
    ax.get_yticklabels()[0].set_fontweight("bold")
    lo, hi = np.floor(base.min() * 10) / 10, np.ceil(final.max() * 10) / 10
    xv = nice_ticks(lo, hi, maxn=4)
    ax.set_xticks(xv, [fmt(v, 1) if v else "0" for v in xv])
    ax.set_xlim(lo - 0.01, hi + 0.01)
    ax.set_ylim(-0.6, yy[-1] + 0.6)
    ax.set_xlabel("test r")
    cat_axis(ax)
    cv.key(1.5, 59.0, [("LAG9 ridge", BASE, "o"), ("final", OURS, "o")], gap=3)
    h1 = rep["H1"]
    g = (final - base)[order]
    axg = cv.ax(42, BOT, 11, TOP - BOT)
    axg.barh(yy, g, height=0.62, color=ACC, alpha=0.85)
    axg.errorbar(h1["mean_diff"], 0, xerr=[[h1["mean_diff"] - h1["ci"][0]], [h1["ci"][1] - h1["mean_diff"]]],
                 fmt="s", ms=4, color=ACC, capsize=0, lw=1.2)
    gmax = max(0.04, np.ceil(g.max() * 50) / 50)
    axg.set_ylim(ax.get_ylim())
    axg.set_xlim(min(0, g.min() * 1.1), gmax)
    axg.set_xticks([0, gmax], ["0", fmt(gmax, 2)])
    axg.set_yticks([])
    axg.spines["left"].set_visible(False)
    axg.axvline(0, color=INK2, lw=0.6)
    axg.axhline(0.7, color=MUTE, lw=0.6)
    axg.set_xlabel("gain")
    cv.letter(1, 63.5, "a", "H1 on ScaleOLDNEW")
    nums["a"] = {"n": len(subs), "mean_base": float(base.mean()), "mean_final": float(final.mean()),
                 "gain_min": float(g.min()), "gain_max": float(g.max()), "H1_report": {q: w for q, w in h1.items()
                                                                                      if q != "per_patient"}}

    # ---------------- b: each development finding in both cohorts (95 % CI); interaction on its own scale
    rows = [("h1", "H1  final − LAG9 ridge", True), ("h2", "H2  learned − fixed", True),
            ("fixed1", "final − fixed bank", True), ("half", "half calibration", False)]
    bx, bw = 85.0, 25.0

    def draw(axx, items, y0):
        for j, (k, lab, ok) in enumerate(items):
            y = y0 - j
            for dy, src, col in ((0.17, dev.get(k), DEV), (-0.17, ext.get(k), EXT)):
                if src:
                    axx.plot(src["ci"], [y + dy] * 2, color=col, lw=1.1, solid_capstyle="butt", zorder=2)
                    axx.plot(src["mean_diff"], y + dy, "o", ms=3.8, color=col, mec="white", mew=0.5, zorder=3)
        axx.axvline(0, color=INK2, lw=0.6, zorder=0)
        axx.set_yticks([y0 - j for j in range(len(items))], [it[1] for it in items])
        for t, it in zip(axx.get_yticklabels(), items):
            t.set_color(INK if it[2] else ACC)
        cat_axis(axx)
    hb = 33.0                                            # height of the 4-row block (mm)
    axb = cv.ax(bx, TOP - hb, bw, hb)
    draw(axb, rows, 3)
    allx = [v for k, _, _ in rows for d in (dev, ext) if d.get(k) for v in d[k]["ci"]]
    x0, x1 = np.floor(min(allx + [0]) * 50) / 50, np.ceil(max(allx) * 50) / 50
    axb.set_xlim(x0 - 0.004, x1 + 0.004)
    xv = nice_ticks(x0, x1, maxn=3)
    axb.set_xticks(xv, [fmt(v, 2) if v else "0" for v in xv])
    axb.set_ylim(-0.5, 3.5)
    axb.axhline(0.5, color=MUTE, lw=0.6)
    axi = cv.ax(bx, BOT, bw, 7.5)
    draw(axi, [("inter", "target × reference", False)], 0)
    allx = [v for d in (dev, ext) if d.get("inter") for v in d["inter"]["ci"]]
    if allx:
        xi1 = np.ceil(max(allx) * 10) / 10
        axi.set_xlim(min(-0.05, min(allx) - 0.02), xi1 + 0.01)
        xv = np.round(np.arange(0, xi1 + 1e-9, 0.1), 1)
        axi.set_xticks(xv, [fmt(v, 1) if v else "0" for v in xv])
    axi.set_ylim(-0.5, 0.5)
    cv.text(bx + bw / 2, 3.0, "difference in test r", ha="center", va="baseline")
    cv.key(60.0, 59.0, [("ds004752", DEV, "o"), ("ScaleOLDNEW", EXT, "o")], gap=3)
    cv.letter(58.5, 63.5, "b", "Replication")

    # ---------------- c: scalp-iEEG common mode vs target x reference interaction, per patient
    axc = cv.ax(124, BOT, 25, TOP - BOT - 7.0)
    for s, (x, it) in cmds.items():
        axc.plot(x, it, "o", ms=3.6, color=DEV, mec="white", mew=0.4, alpha=0.9, zorder=2)
    for s, v in cm17.items():
        axc.plot(abs(v["r_scalp_cm_vs_ieeg_cm"]), it17[s], "o", ms=3.8, color=EXT, mec="white", mew=0.4, zorder=3)
    THR = 0.6                                            # "same signal" threshold for the labels
    hd = sorted((it, s.replace("sub-", "")) for s, (x, it) in cmds.items() if x > THR)
    he = [(abs(v["r_scalp_cm_vs_ieeg_cm"]), it17[s], s.replace("sub-", "")) for s, v in cm17.items()
          if abs(v["r_scalp_cm_vs_ieeg_cm"]) > THR]
    for x, it, _ in [(cmds["sub-" + s][0], it, s) for it, s in hd] + he:    # ring the patients above threshold
        axc.plot(x, it, "o", ms=7.0, mfc="none", mec=ACC, mew=0.8, zorder=4)
    kx = 114.5                                           # key above the axes: ring = |r| above threshold
    cv.fig.lines.append(Line2D([(kx + 1.0) / cv.w], [(TOP - 0.2) / cv.h], marker="o", ms=6, mfc="none", mec=ACC,
                               mew=0.8, transform=cv.fig.transFigure))
    cv.text(kx + 2.8, TOP - 1.2, f"|r| > {THR}", color=ACC, va="baseline")
    t_ = cv.text(kx + 2.8, TOP - 4.8, ", ".join(sorted(s for _, s in hd)) or "none", color=DEV, va="baseline")
    r_ = cv.fig.canvas.get_renderer()
    cv.text(kx + 2.8 + t_.get_window_extent(r_).width / cv.fig.dpi * 25.4 + 2.0, TOP - 4.8,
            ", ".join(sorted(s for _, _, s in he)) or "none", color=EXT, fontweight="bold", va="baseline")
    axc.axhline(0, color=INK2, lw=0.6, zorder=0)
    axc.set_xlim(-0.03, 1.03)
    axc.set_xticks([0, 0.5, 1.0], ["0", "0.5", "1"])
    yt = [v for v in axc.get_yticks() if axc.get_ylim()[0] <= v <= axc.get_ylim()[1]]
    axc.set_yticks(yt, [fmt(v, 1) if abs(v) > 1e-9 else "0" for v in yt])
    axc.set_xlabel("common-mode |r|")
    axc.set_ylabel("target × reference")
    nums["c_counts"] = {"threshold": THR, "ds004752": [len(hd), len(cmds)], "ScaleOLDNEW": [len(he), len(cm17)]}
    cv.letter(113, 63.5, "c", "Common mode")
    nums["c"] = {"ds004752": {s: list(v) for s, v in cmds.items()},
                 "ScaleOLDNEW": {s: [abs(v["r_scalp_cm_vs_ieeg_cm"]), it17[s]] for s, v in cm17.items()}}
    save(cv, out, "fig5", nums)


# ----------------------------------------------------------------------------- supplementary figures
def _spread(ys, gap):
    """Push label positions apart (order kept) so that neighbours are at least `gap` apart."""
    o = np.argsort(ys)
    y = np.asarray(ys, float)[o].copy()
    for _ in range(100):
        moved = False
        for i in range(1, len(y)):
            if y[i] - y[i - 1] < gap - 1e-12:
                d = (gap - (y[i] - y[i - 1])) / 2
                y[i - 1] -= d
                y[i] += d
                moved = True
        if not moved:
            break
    out = np.empty_like(y)
    out[o] = y
    return out


def _mixed_slope(rows):
    """Slope of r on session gap with a random intercept per patient (statsmodels); OLS if unavailable."""
    G = np.array([r[1] for r in rows], float)
    V = np.array([r[2] for r in rows], float)
    try:
        import pandas as pd
        import statsmodels.formula.api as smf
        df = pd.DataFrame(rows, columns=["subject", "gap", "r"])
        fit = smf.mixedlm("r ~ gap", df, groups=df["subject"]).fit()
        return float(fit.params["gap"]), float(fit.params["Intercept"]), fit.conf_int().loc["gap"].tolist(), "mixed"
    except Exception:
        s, i = np.polyfit(G, V, 1)
        return float(s), float(i), [float("nan"), float("nan")], "ols"


def _contacts(D):
    out = {}
    for s, d in D.items():
        m = d["fracs"]["0.2"]["models"]["combo_nosess"]
        cv, te = np.array(m["cv_contacts"], float), np.array(m["contacts"], float)
        ok = np.isfinite(cv) & np.isfinite(te)
        out[s] = (cv[ok], te[ok])
    return out


def figS1(runs, out):
    """Contact reliability (primary), coverage (secondary), longevity across sessions (secondary)."""
    from scipy.stats import spearmanr
    main_run = os.path.join(runs, "shared_bip_car")
    P1 = _contacts(load_strict(main_run))
    P2 = _contacts(load_dir(os.path.join(main_run, "readout_combo5_strict"), "readout_"))
    if not P1 or not P2:
        raise FileNotFoundError(2, "missing", os.path.join(main_run, "readout_combo5_strict"))
    cv_ = Canvas(W2, 64)
    TOP, BOT = 53.0, 10.5
    nums = {}

    # ---------------- a: calibration cross-validation r vs test r, every contact (primary level)
    subs = sorted(P1)
    rho = np.array([spearmanr(*P1[s]).correlation for s in subs])
    cv = np.concatenate([P1[s][0] for s in subs])
    te = np.concatenate([P1[s][1] for s in subs])
    side = TOP - BOT
    ax = cv_.ax(11, BOT, side, side)
    lim = (np.floor(min(np.percentile(cv, 0.3), np.percentile(te, 0.3)) * 10) / 10,
           np.ceil(max(np.percentile(cv, 99.8), np.percentile(te, 99.8)) * 10) / 10)
    ax.plot(lim, lim, color=MUTE, lw=0.6, zorder=0)
    ax.scatter(cv, te, s=3, color=OURS, alpha=0.3, lw=0, zorder=2)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    tk = nice_ticks(*lim, maxn=5)
    ax.set_xticks(tk, [fmt(v, 1) if v else "0" for v in tk])
    ax.set_yticks(tk, [fmt(v, 1) if v else "0" for v in tk])
    ax.set_xlabel("calibration r")
    ax.set_ylabel("test r")
    ax.text(0.04, 0.97, f"Spearman per patient\nmedian {np.median(rho):.2f} ({rho.min():.2f}–{rho.max():.2f})",
            transform=ax.transAxes, va="top", color=OURS, linespacing=1.2)
    cv_.letter(1, 58.5, "a", "Reliability")
    nums["a"] = {"level": "primary, strict", "n_contacts": int(len(cv)), "rho_median": float(np.median(rho)),
                 "rho_mean": float(rho.mean()), "rho_range": [float(rho.min()), float(rho.max())],
                 "rho_pooled": float(spearmanr(cv, te).correlation)}

    # ---------------- b: coverage, secondary level: mean test r of the contacts kept, most reliable first
    cv = np.concatenate([P2[s][0] for s in sorted(P2)])
    te = np.concatenate([P2[s][1] for s in sorted(P2)])
    order = np.argsort(-cv)
    kept = np.arange(1, len(cv) + 1) / len(cv) * 100
    cum = np.cumsum(te[order]) / np.arange(1, len(cv) + 1)
    k0 = 10
    axb = cv_.ax(72, BOT, 34, side)
    axb.plot(kept[k0:], cum[k0:], color=OURS, lw=1.4, zorder=3)
    axb.axhline(te.mean(), color=INK2, lw=0.6, ls=(0, (2, 1.5)), zorder=1)
    nums["b"] = {"level": "secondary, strict, final model", "n_contacts": int(len(cv)),
                 "all_contacts_mean": float(te.mean()), "thresholds": {}}
    rowsT = []
    for thr in (0.3, 0.4, 0.5):
        sel = cv > thr
        if sel.sum() < 10:
            continue
        x, y = sel.mean() * 100, te[sel].mean()
        axb.plot(x, y, "o", ms=4, color=ACC, mec="white", mew=0.5, zorder=4)
        rowsT.append((thr, x, y))
        nums["b"]["thresholds"][str(thr)] = {"kept_pct": float(x), "mean_test_r": float(y)}
    axb.set_xlim(0, 100)
    axb.set_xticks([0, 50, 100])
    ylo, yhi = te.mean() - 0.04, cum[k0:].max() + 0.02
    axb.set_ylim(ylo, yhi)
    yv = nice_ticks(ylo, yhi, maxn=4)
    axb.set_yticks(yv, [fmt(v, 1) for v in yv])
    axb.set_xlabel("contacts kept (%)")
    axb.set_ylabel("mean test r of kept contacts")
    axb.text(0.04, (te.mean() - ylo) / (yhi - ylo) - 0.02, f"all contacts {fmt(te.mean(), 2)}",
             transform=axb.transAxes, ha="left", va="top", color=INK2)
    # threshold table in the empty lower-left corner (few contacts kept, low r)
    dl = 3.3 / side
    for xc, head, cell in ((0.50, "cal. r", lambda q: f"> {q[0]:.1f}"), (0.75, "kept", lambda q: f"{q[1]:.0f} %"),
                           (0.98, "test r", lambda q: fmt(q[2], 2))):          # right-aligned columns
        axb.text(xc, 0.97, head, transform=axb.transAxes, color=INK2, ha="right", va="top")
        for j, q in enumerate(rowsT):
            axb.text(xc, 0.97 - (j + 1) * dl, cell(q), transform=axb.transAxes, color=ACC, ha="right", va="top")
    cv_.letter(61, 58.5, "b", "Coverage")

    # ---------------- c: longevity across sessions (secondary level, session-wise normalisation)
    lon = load_dir(os.path.join(main_run, "readout_longevity"), "readout_")
    axc = cv_.ax(122, BOT, 26, side)
    rows, rows_adv = [], []
    for d in lon.values():
        fr, order_ = d["fracs"]["0.2"], d["session_order"]
        last = order_.index(fr["calib_sessions"][-1])
        mods = fr["models"]
        pts = sorted((order_.index(s) - last, v, mods.get("base", {}).get("sessions", {}).get(s))
                     for s, v in mods["combo"].get("sessions", {}).items()
                     if d["test_trials_per_session"].get(s, 0) >= 5)
        if len(pts) >= 2:
            axc.plot([p[0] for p in pts], [p[1] for p in pts], color=MUTE, lw=0.7, zorder=1)
        rows += [(d["subject"], p[0], p[1]) for p in pts]
        rows_adv += [(d["subject"], p[0], p[1] - p[2]) for p in pts if p[2] is not None]
    nums["c"] = {"n_patient_sessions": len(rows)}
    if rows:
        slope, icpt, ci, kind = _mixed_slope(rows)
        G = np.array([r[1] for r in rows])
        gx = np.array([G.min(), G.max()])
        axc.plot(gx, icpt + slope * gx, color=OURS, lw=1.6, zorder=3)
        axc.text(0.04, 0.04, f"{fmt(slope, sign=True)}\nper session", transform=axc.transAxes, ha="left",
                 va="bottom", color=OURS, fontweight="bold", linespacing=1.15)
        nums["c"].update({"slope": slope, "intercept": icpt, "slope_ci": ci, "model": kind})
        if rows_adv:
            sa, _, cia, _ = _mixed_slope(rows_adv)
            nums["c"].update({"advantage_slope": sa, "advantage_slope_ci": cia})
        axc.set_xticks(range(int(G.min()), int(G.max()) + 1, 2))
        axc.set_xlim(G.min() - 0.3, G.max() + 0.3)
    yv = nice_ticks(*axc.get_ylim(), maxn=4)
    axc.set_yticks(yv, [fmt(v, 1) for v in yv])
    axc.set_xlabel("session gap")
    axc.set_ylabel("test r")
    cv_.letter(112, 58.5, "c", "Longevity")
    save(cv_, out, "figS1", nums)


BANDS = {"delta": "δ", "theta": "θ", "alpha": "α", "beta": "β"}


def figS2(runs, out):
    """Band power of a new patient's contacts without their SEEG (secondary level, leave one patient out)."""
    p = os.path.join(runs, "zeroshot_power", "zeroshot_subjects.json")
    if not os.path.exists(p):
        raise FileNotFoundError(2, "missing", p)
    zs = json.load(open(p))
    bands = list(zs[0]["bands"])
    rows = [("zs_global", "global scalp average", BASE_PT, (0, (1.2, 1.4)), 0),
            ("zs_nearest", "nearest scalp channel", BASE, (0, (1.2, 1.4)), 0),
            ("zs_geo_shuf", "positions shuffled", INK2, (0, (3, 1.6)), 0),
            ("cal_ridge", "ridge, 4 min SEEG", FIXED_TXT, "-", 1),
            ("zs_geo", "zero-shot, no SEEG", ACC, "-", 2),
            ("zs_cal", "zero-shot + ridge", OURS, "-", 1)]
    get = lambda m, b: np.array([r["bands"][b][m]["r_induced"] for r in zs], float)   # noqa: E731
    cv = Canvas(W2, 62)
    TOP, BOT = 52.0, 9.0
    nums = {"n": len(zs), "a": {}}

    ax = cv.ax(12, BOT, 62, TOP - BOT)
    xb = np.arange(len(bands), dtype=float)
    ends = []
    dodge = np.linspace(-0.08, 0.08, len(rows))
    for (m, lab, col, ls, emph), dx in zip(rows, dodge):
        mu = np.array([get(m, b).mean() for b in bands])
        ci = np.array([boot_ci(get(m, b), f"figS2 {m} {b}") for b in bands])
        ax.plot(xb + dx, mu, color=col, lw=(0.9, 1.1, 1.7)[emph], ls=ls, zorder=2 + emph)
        for x, (l_, h_) in zip(xb + dx, ci):
            ax.plot([x, x], [l_, h_], color=col, lw=0.6, alpha=0.8, zorder=2 + emph, solid_capstyle="butt")
        ax.plot(xb + dx, mu, "o", ms=(3.0, 3.4, 4.0)[emph], color=col, mec="white", mew=0.4, zorder=3 + emph)
        ends.append(mu[-1])
        nums["a"][m] = {b: {"mean": float(u), "ci": list(c)} for b, u, c in zip(bands, mu, ci)}
    yl, yh = ax.get_ylim()
    ly = _spread(ends, 3.3 / (TOP - BOT) * (yh - yl))
    for (m, lab, col, ls, emph), y0, y1 in zip(rows, ends, ly):
        ax.plot([xb[-1] + 0.14, xb[-1] + 0.24], [y0, y1], color=col, lw=0.5)
        ax.text(xb[-1] + 0.27, y1, lab, color=col, va="center", fontweight="bold" if emph == 2 else "normal")
    ax.set_xticks(xb, [BANDS.get(b, b) for b in bands])
    ax.tick_params(axis="x", length=0, pad=3)
    ax.spines["bottom"].set_visible(False)
    ax.set_xlim(-0.3, len(bands) - 1 + 1.45)
    ax.set_ylim(min(yl, min(ly)) , max(yh, max(ly)))
    yv = nice_ticks(*ax.get_ylim(), maxn=4)
    ax.set_yticks(yv, [tl(v) for v in yv])
    ax.set_ylabel("r of log band power")
    cv.letter(1, 57.0, "a", "Band power without SEEG")

    zg = np.mean([get("zs_geo", b) for b in bands], 0)
    cr = np.mean([get("cal_ridge", b) for b in bands], 0)
    side = TOP - BOT
    axb = cv.ax(150 - 3 - side, BOT, side, side)
    lim = (np.floor(min(zg.min(), cr.min()) * 20) / 20, np.ceil(max(zg.max(), cr.max()) * 20) / 20)
    axb.plot(lim, lim, color=MUTE, lw=0.6, zorder=0)
    axb.plot(cr, zg, "o", ms=4, color=ACC, mec="white", mew=0.5, zorder=3)
    axb.set_xlim(lim)
    axb.set_ylim(lim)
    tk = nice_ticks(*lim, maxn=4)
    axb.set_xticks(tk, [tl(v) for v in tk])
    axb.set_yticks(tk, [tl(v) for v in tk])
    axb.set_xlabel("ridge, 4 min SEEG")
    axb.set_ylabel("zero-shot, no SEEG")
    above = int((zg > cr).sum())
    axb.text(0.04, 0.97, f"{above}/{len(zg)} above the diagonal", transform=axb.transAxes, va="top", color=ACC)
    cv.letter(150 - 3 - side - 13, 57.0, "b", "Per patient")
    nums["b"] = {"zero_shot_mean": float(zg.mean()), "ridge_mean": float(cr.mean()), "zero_shot_above": above,
                 "per_patient_mean_over_bands": True}
    save(cv, out, "figS2", nums)


def figS3(runs, out):
    """ScaleOLDNEW scalp-channel audit: raw amplitude vs similarity to amplifier inputs without a 10-05 name."""
    import csv
    p = os.path.join(runs, "chanqc20", "chanqc20_channels_v20_1.csv")
    if not os.path.exists(p):
        raise FileNotFoundError(2, "missing", p)
    rows = [r for r in csv.DictReader(open(p, encoding="utf-8")) if r["group"] != "no 10-05 name"]

    def num(r, k):
        try:
            return float(r[k])
        except (TypeError, ValueError):
            return np.nan
    groups = [("kept, listed in the authors' files", lambda g: g == "kept, in .set", BASE_PT, 3.0, 0.9),
              ("kept, not listed", lambda g: g == "kept, not in .set", OURS, 4.2, 1.0),
              ("not kept (bad channel)", lambda g: g.startswith("not kept"), ACC, 4.2, 1.0)]
    cv = Canvas(W1, 76)
    ax = cv.ax(13, 10, 69, 50)
    nums = {}
    for lab, fn, col, ms, a in groups:
        R_ = [r for r in rows if fn(r["group"])]
        x = np.array([num(r, "raw_sd_uV") for r in R_])
        y = np.clip(np.array([num(r, "r_unnamed_max") for r in R_]), 1e-3, 0.99985)
        ok = np.isfinite(x) & np.isfinite(y)
        ax.plot(x[ok], y[ok], "o", ms=ms, color=col, alpha=a, mec="white", mew=0.3, ls="none",
                zorder=2 if col == BASE_PT else 3)
        nums[lab] = {"n": int(ok.sum()), "raw_sd_gt500_and_r_ge099": int(((x > 500) & (y >= 0.99)).sum()),
                     "max_raw_sd": float(np.nanmax(x)) if len(x) else None}
    ax.set_xscale("log")
    ax.set_yscale("logit")
    ally = np.concatenate([l.get_ydata() for l in ax.get_lines()])
    y0 = 10 ** np.floor(np.log10(max(1e-3, np.nanmin(ally))) - 1e-9)       # lowest decade with data
    ax.set_ylim(y0 * 0.6, 0.9999)
    yt = [v for v in (0.001, 0.01, 0.1, 0.5, 0.9, 0.99, 0.999) if v >= y0]
    ax.set_yticks(yt, [tl(v, 3) for v in yt])
    ax.minorticks_off()
    ax.set_xticks([10, 100, 1000], ["10", "100", "1000"])
    ax.axvline(500, color=INK2, lw=0.6, ls=(0, (2, 1.5)), zorder=1)
    ax.axhline(0.99, color=INK2, lw=0.6, ls=(0, (2, 1.5)), zorder=1)
    ax.set_xlabel("raw standard deviation (µV)")
    ax.set_ylabel("max |r| with unnamed inputs")
    keys = [(f"{lab} ({nums[lab]['n']})", col, "o") for lab, fn, col, ms, a in groups]   # key above the axes
    cv.key(1, 65.0, keys[:1], text_colors={keys[0][0]: INK2})
    cv.key(1, 61.6, keys[1:], gap=4)
    cv.text(1, 71.5, "ScaleOLDNEW scalp channels, patients 01–07", fontsize=FS_T, va="baseline")
    save(cv, out, "figS3", nums)


def fig6(runs, out):
    """(a) Recorded SEEG and predictions of the final model and the LAG9 ridge for the best, median and worst contact
    of the median patient (a 4-s window of the middle test trial); (b) coherence between prediction and recording,
    mean over patients with patient-bootstrap 95 % CI."""
    import glob
    d = os.path.join(runs, "shared_bip_car", "readout_combo5_strict" + STRICT + "_pred")
    files = sorted(glob.glob(os.path.join(d, "pred_sub-*.npz")))
    if not files:
        raise FileNotFoundError(2, "missing", os.path.join(d, "pred_sub-*.npz"))
    Z = {os.path.basename(p)[5:-4]: np.load(p) for p in files}
    rmean = {s: float(np.nanmean(z["r_final"])) for s, z in Z.items()}
    med = float(np.median(list(rmean.values())))
    sid = min(rmean, key=lambda s: (abs(rmean[s] - med), s))           # patient closest to the median
    z = Z[sid]
    tl_ = z["trial_len"].astype(int)
    m = len(tl_) // 2                                                 # middle test trial, first 4 s
    s0 = int(tl_[:m].sum())
    n = min(int(tl_[m]), 800)
    tt = np.arange(n) / 200.0
    nums = {"patient": sid, "patient_mean_r_final": rmean[sid], "median_over_patients": med, "trial": int(m),
            "window_s": n / 200.0, "contacts": []}
    cv = Canvas(W2, 78)
    labs = ("best contact", "median contact", "worst contact")
    top, hh, gap = 62.0, 15.5, 3.0
    for j, lab in enumerate(labs):
        k = int(z["pick"][j])
        ax = cv.ax(25, top - j * (hh + gap) - hh, 72, hh)
        y = z["sig_target"][j, s0:s0 + n].astype(float)
        ax.plot(tt, y, color=INK, lw=0.7, zorder=3)
        ax.plot(tt, z["sig_base"][j, s0:s0 + n].astype(float), color=BASE, lw=0.7, zorder=2)
        ax.plot(tt, z["sig_final"][j, s0:s0 + n].astype(float), color=OURS, lw=0.9, zorder=4)
        lim = np.nanpercentile(np.abs(y), 99.5) * 1.15
        ax.set_ylim(-lim, lim)
        ax.set_xlim(0, tt[-1])
        ax.set_yticks([])
        ax.spines["left"].set_visible(False)
        if j < 2:
            ax.set_xticks([])
            ax.spines["bottom"].set_visible(False)
        else:
            ax.set_xlabel("time in test trial (s)")
        rf, rb = float(z["r_final"][k]), float(z["r_base"][k])
        cv.text(1.0, top - j * (hh + gap) - hh / 2 + 1.2, lab, color=INK, va="center")
        cv.text(1.0, top - j * (hh + gap) - hh / 2 - 2.6, f"r {fmt(rf)} / {fmt(rb)}", color=INK2, va="center")
        nums["contacts"].append({"label": lab, "index": k, "r_final": rf, "r_base": rb})
    cv.letter(1, 73.0, "a", "Recorded and predicted SEEG")
    cv.key(25, 66.8, [("recording", INK, "_"), ("final model", OURS, "_"), ("LAG9 ridge", BASE, "_")], gap=4,
           text_colors={"recording": INK})
    # ---- b: coherence spectrum over patients
    fr = Z[sid]["freqs"]
    sel = (fr >= 1) & (fr <= 70)
    axb = cv.ax(112, 10, 35, 52)
    nums["coherence"] = {}
    for key, col, lab in (("base", BASE, "LAG9 ridge"), ("final", OURS, "final model")):
        C = np.array([zz[f"coh_{key}"] for zz in Z.values()])[:, sel]
        mu = np.nanmean(C, 0)
        ci = np.array([boot_ci(C[:, i], f"fig6 coherence {key} {fr[sel][i]:.2f}") for i in range(C.shape[1])])
        axb.fill_between(fr[sel], ci[:, 0], ci[:, 1], color=col, alpha=0.18, lw=0)
        axb.plot(fr[sel], mu, color=col, lw=1.2)
        nums["coherence"][lab] = {"freqs": fr[sel].tolist(), "mean": mu.tolist()}
    axb.set_xscale("log")
    axb.set_xticks([1, 4, 10, 30, 70], ["1", "4", "10", "30", "70"])
    axb.minorticks_off()
    axb.set_xlim(1, 70)
    ytop = float(axb.get_ylim()[1]) * 1.02
    yt = nice_ticks(0, ytop)
    axb.set_yticks(yt, [tl(v) for v in yt])
    axb.set_ylim(0, ytop)
    axb.set_xlabel("frequency (Hz)")
    axb.set_ylabel("coherence")
    cv.letter(103, 73.0, "b", "Coherence")
    nums["n_patients"] = len(Z)
    save(cv, out, "fig6", nums)


FIGS = {"fig1": fig1, "fig2": fig2, "fig3": fig3, "fig4": fig4, "fig5": fig5, "fig6": fig6,
        "figS1": figS1, "figS2": figS2, "figS3": figS3}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--only", nargs="*", default=None)
    ap.add_argument("--data", default=None, help="ds004752 data folder (s2i_v1) for the real inputs of Fig. 1")
    a = ap.parse_args()
    global DATA_DIR
    DATA_DIR = a.data
    setup()
    for k, fn in FIGS.items():
        if a.only and k not in a.only:
            continue
        print(k)
        try:
            fn(a.runs, a.out)
        except FileNotFoundError as e:
            print(f"  skipped: missing {e.filename}")
        except Exception:                                  # keep going; the traceback says what to fix
            import traceback
            traceback.print_exc()
            print(f"  FAILED: {k}")


if __name__ == "__main__":
    main()

## 3. Vẽ

In [ ]:
#@title 3. Vẽ (để trống ONLY = vẽ mọi hình đã có)
ONLY = ''   #@param {type:"string"}
only = f'--only {ONLY}' if ONLY.strip() else ''
!python -m s2i.figs11 --runs "{RUNS_DIR}" --data "{ROOT}/data/s2i_v1" --out "{FIG_DIR}" {only}

In [ ]:
#@title 4. Xem hình
from IPython.display import Image, display
for p in sorted(glob.glob(f'{FIG_DIR}/*.png')):
    print(os.path.basename(p))
    display(Image(p, width=1100))

In [ ]:
#@title 5. Gói PNG + numbers.json để gửi lại
import shutil, tempfile
from google.colab import files
tmp = tempfile.mkdtemp()
for f in glob.glob(f'{FIG_DIR}/*.png') + glob.glob(f'{FIG_DIR}/*_numbers.json'):
    shutil.copy(f, tmp)
z = shutil.make_archive('/content/s2i_figs11', 'zip', tmp)
print(z, round(os.path.getsize(z) / 1e6, 1), 'MB')
files.download(z)

## Gửi lại
- Output ô 3 (dòng `layout:` cho biết số chữ chồng nhau và vùng trống lớn nhất).
- File `s2i_figs11.zip` (ô 5), hoặc chỉ ảnh PNG nếu tiện.